# 🎯 [DAT09] 군수데이터 전략기획·환경평가 대시보드
### Google Colab 실습 환경 (데이터·자동화·업무플랫폼 / Level 2)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/DAT09/DAT09_solution.ipynb)

---

## 📌 1. 프로젝트 개요 및 배경

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 데이터·자동화·업무플랫폼 (Level 2) |
| **학습 소요 시간** | 10개 미션 (5일 / 총 27.0시간) |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
미국 국방부(DoD)는 군수 데이터 전략과 환경 평가를 통해 서로 다른 체계에 흩어진 데이터의 중복을 없애고 표준 키와 코드를 정해 업무 중심으로 통합하려는 접근을 공개해 왔다. 이 사례의 학습 초점은 중복 제거, 표준 키·코드, 업무 중심 통합, 역할별 화면, 자동 갱신, 재사용성이다. 본 프로젝트는 이를 바탕으로 교육용 가상 보급·정비·탄약 체계 3종의 재고 보고를 코드로 만들어 통합 대시보드로 가시화한다. 학습자는 연동 규격 분석, 코드 표준화와 결측 전처리, 통합 저장소와 검증 룰, 변경 이벤트 동기화, 불일치 식별과 품질 지수, 역할별 상황판, 데이터관리관 승인, 대량 트랜잭션 시험, AAR 재튜닝, 최종 통합을 차례로 수행한다. 최종 목표는 마스터 데이터의 확정은 사람이 승인하도록 하면서 이기종 군수 정보를 한 화면에서 보는 통합 군수 상황판을 완성하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---



In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 01. 이기종 군수 정보체계(DELIIS, 장비정비정보체계, 탄약체계) 데이터 연동 규격 분석
**소요시간**: 약 1.5시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

이기종 군수 정보체계(DELIIS, 장비정비, 탄약체계) 데이터 연동 규격을 분석한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **이기종 군수 정보체계 3종의 품목코드·부대코드·날짜·수량 연동 규격을 비교 분석하고, 통합 군수 상황 파악 업무를 AS-IS/TO-BE로 비교한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **가상의 보급·정비·탄약 체계가 같은 재고를 서로 다른 코드와 날짜 형식으로 보고하는 자료를 코드로 작성하여 원인을 분석하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `docs/dat09_as_is_to_be_spec.md` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 현행 업무(AS-IS)와 개선 업무(TO-BE) 비교의 명확성

#### 미션 PASS 조건

- 정량 기준: 업무 단계 분해 10단계 이상, 병목 구간 3개소 이상 도출


- 진행 기준: 업무 정의서 및 입출력 데이터 명세 작성 완료


- 검증 방법: 실무 프로세스와의 일치성 검증


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""가상 이기종 군수정보체계 3종(보급·정비·탄약)의 품목코드·부대코드·날짜·수량 연동 규격을 비교 분석해 코드 변환 규칙을 정의하고, 통합 군수 상황 파악 업무를 AS-IS/TO-BE 단계로 분해해 병목 구간을 도출한다."""
import datetime
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 이기종 군수정보체계 3종(A 보급, B 정비, C 탄약)의 재고 보고 생성기
# 실제 체계명·부대·품목·수량 없음. 같은 (품목, 부대) 재고를 세 체계가 서로 다른 코드·날짜 형식으로 보고한다.
# 기준 날짜: 2026-09-30(가상). 마스터(정답) 재고 TRUE 와 체계별 보고에는 의도적 불일치·결측이 섞여 있다.
# ---------------------------------------------------------------------------
N_PART, N_UNIT = 60, 12
CATS = ["급식", "피복", "유류", "정비소모", "수리부속", "탄약"]
PARTS = {p: {"name": f"가상품목{p:02d}", "cat": CATS[(p - 1) % 6]} for p in range(1, N_PART + 1)}
FMT = {"A": dict(part=lambda p: f"AP-{p:05d}", unit=lambda u: f"U{u:02d}", date=lambda d: d.strftime("%Y%m%d")),
       "B": dict(part=lambda p: f"{p:05d}-X", unit=lambda u: f"가상{u:02d}대대", date=lambda d: d.strftime("%Y-%m-%d")),
       "C": dict(part=lambda p: f"P{p}", unit=lambda u: str(u), date=lambda d: d.strftime("%d/%m/%Y"))}
BASE_DATE = datetime.date(2026, 9, 30)


def make_systems(seed=9, mismatch=0.10, missing=0.03):
    """TRUE(정답 재고)와 체계별 원본 보고 레코드(체계 고유 형식)를 만든다."""
    rng = np.random.default_rng(seed)
    true = {(p, u): int(rng.integers(20, 500)) for p in PARTS for u in range(1, N_UNIT + 1)}
    out = {s: [] for s in "ABC"}
    truth_diff = {}                                   # (체계, p, u) -> 불일치 원인(정답)
    for (p, u), q in true.items():
        for s in "ABC":
            qty, cause = q, None
            r = rng.random()
            if r < mismatch / 3:                      # 입력 오류(자릿수/오타)
                qty, cause = q + int(rng.choice([-1, 1]) * rng.integers(10, 60)), "입력 오류"
            elif r < 2 * mismatch / 3:                # 반영 지연(전일 값)
                qty, cause = q + int(rng.integers(5, 40)), "반영 지연"
            elif r < mismatch:                        # 단위 오류(10배)
                qty, cause = q * 10, "단위 오류"
            d = BASE_DATE - datetime.timedelta(days=int(cause == "반영 지연"))
            if rng.random() < missing:                # 결측
                qty, cause = None, "결측"
            if cause:
                truth_diff[(s, p, u)] = cause
            out[s].append({"part": FMT[s]["part"](p), "unit": FMT[s]["unit"](u), "date": FMT[s]["date"](d), "qty": qty})
    return true, out, truth_diff


def parse_part(code):
    # [TODO : 학생 실습 1] 이기종 군수 정보체계(DELIIS, 장비정비정보체계, 탄약체계) 데이터 연동 규격 분석 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def parse_unit(code):
    # [TODO : 학생 실습 1] 이기종 군수 정보체계(DELIIS, 장비정비정보체계, 탄약체계) 데이터 연동 규격 분석 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def parse_date(s):
    if "/" in s:
        d, m, y = s.split("/"); return datetime.date(int(y), int(m), int(d))
    if "-" in s:
        return datetime.date(*map(int, s.split("-")))
    return datetime.date(int(s[:4]), int(s[4:6]), int(s[6:]))

true, raw, tdiff = make_systems()

# 1) 체계별 연동 규격 비교표(필드 형식 실제 샘플에서 추출)
spec = pd.DataFrame({"체계": ["A(보급)", "B(정비)", "C(탄약)"],
                     "품목코드 예": [raw[s][0]["part"] for s in "ABC"], "부대코드 예": [raw[s][0]["unit"] for s in "ABC"],
                     "날짜 예": [raw[s][0]["date"] for s in "ABC"], "수량 필드": ["qty(정수)"] * 3,
                     "갱신 주기": ["일 1회", "수시", "주 1회"], "결측 표기": ["빈칸", "null", "-1"]})
print("[체계별 연동 규격 비교] [교육용가상자료]")
print(spec.to_string(index=False))

# 2) 변환 규칙 검증: 세 체계의 코드가 마스터 키로 정확히 1:1 변환되는지
rule_ok = all(parse_part(FMT[s]["part"](p)) == p and parse_unit(FMT[s]["unit"](u)) == u for s in "ABC" for p in PARTS for u in range(1, N_UNIT + 1))
date_ok = all(parse_date(FMT[s]["date"](BASE_DATE)) == BASE_DATE for s in "ABC")
print(f"\n코드 변환 규칙 왕복 검증: 품목 {N_PART}종 x 부대 {N_UNIT}개 x 3체계 {'전부 일치' if rule_ok else '불일치'}, 날짜 형식 3종 {'일치' if date_ok else '불일치'}")

# 3) 연동 현황: 키 일치도·불일치·결측(체계 간 비교)
rows = []
for s in "ABC":
    for r in raw[s]:
        rows.append({"sys": s, "p": parse_part(r["part"]), "u": parse_unit(r["unit"]), "date": parse_date(r["date"]), "qty": r["qty"]})
df = pd.DataFrame(rows)
wide = df.pivot_table(index=["p", "u"], columns="sys", values="qty", aggfunc="first")
miss = wide.isna().sum()
diff = (wide.max(axis=1) != wide.min(axis=1)) & wide.notna().all(axis=1)
print(f"세 체계 보고 {len(df)}건 / 비교 키 {len(wide)}개: 결측 A {miss['A']}·B {miss['B']}·C {miss['C']}건, 체계 간 수량 불일치 키 {int(diff.sum())}개({diff.mean():.1%})")
print("날짜 지연(기준일 이전 보고) 건수:", int((df["date"] < BASE_DATE).sum()))

# 4) 통합 군수 상황 파악 업무 AS-IS / TO-BE (가상 측정 소요 분)
steps = [("체계 A 재고 조회·내려받기", 25, "정해진 시각 자동 수집", 2), ("체계 B 정비 현황 조회·내려받기", 30, "자동 수집", 2),
         ("체계 C 탄약 현황 요청(전화·메일)", 45, "연동 요청 규격화·자동 수집", 5), ("엑셀로 3개 자료 합치기", 60, "표준 키로 자동 병합", 1),
         ("품목코드 수기 대조", 70, "코드 변환표 자동 적용", 1), ("부대명·날짜 형식 통일", 35, "변환 규칙 자동 적용", 1),
         ("결측·이상값 확인(전화 문의)", 55, "결측·불일치 후보 자동 표시", 10), ("체계 간 불일치 원인 확인", 50, "원인 후보 자동 추천", 12),
         ("지표(재고·가동·소모) 집계", 40, "집계 쿼리 자동 실행", 1), ("군수참모 보고용 표·그래프 작성", 65, "상황판 자동 갱신", 2),
         ("마스터 데이터 수정 요청·확정", 45, "데이터관리관 승인 화면", 10), ("보고 후 수정·재배포", 30, "단일 데이터 원천 사용", 3)]
proc = pd.DataFrame(steps, columns=["업무 단계", "AS-IS(분)", "TO-BE 개선", "TO-BE(분)"])
proc.index = range(1, len(proc) + 1)
thr = proc["AS-IS(분)"].mean() + 0.5 * proc["AS-IS(분)"].std()
proc["병목"] = np.where(proc["AS-IS(분)"] >= thr, "병목", "")
print(f"\n[AS-IS / TO-BE 업무 단계 {len(proc)}단계] (병목 기준: 평균+0.5표준편차 = {thr:.0f}분 이상)")
print(proc[["업무 단계", "AS-IS(분)", "TO-BE(분)", "병목"]].to_string())
nb = int((proc["병목"] == "병목").sum())
a, t = proc["AS-IS(분)"].sum(), proc["TO-BE(분)"].sum()
print(f"\n업무 단계 {len(proc)}단계(목표 10단계 이상: {'달성' if len(proc) >= 10 else '미달'}), 병목 {nb}개소(목표 3개소 이상: {'달성' if nb >= 3 else '미달'})")
print(f"1회 상황 파악 AS-IS {a}분 -> TO-BE {t}분 ({1 - t / a:.0%} 단축 예상, 가정값)")


<details>
<summary><b>💡 [참고] 미션 01 예상 실행 결과 (Output)</b></summary>

```text
[체계별 연동 규격 비교] [교육용가상자료]
   체계   품목코드 예 부대코드 예       날짜 예   수량 필드 갱신 주기 결측 표기
A(보급) AP-00001    U01   20260930 qty(정수)  일 1회    빈칸
B(정비)  00001-X 가상01대대 2026-09-30 qty(정수)    수시  null
C(탄약)       P1      1 30/09/2026 qty(정수)  주 1회    -1

코드 변환 규칙 왕복 검증: 품목 60종 x 부대 12개 x 3체계 전부 일치, 날짜 형식 3종 일치
세 체계 보고 2160건 / 비교 키 720개: 결측 A 18·B 14·C 14건, 체계 간 수량 불일치 키 176개(24.4%)
날짜 지연(기준일 이전 보고) 건수: 73

[AS-IS / TO-BE 업무 단계 12단계] (병목 기준: 평균+0.5표준편차 = 53분 이상)
                   업무 단계  AS-IS(분)  TO-BE(분)  병목
1        체계 A 재고 조회·내려받기        25         2    
2     체계 B 정비 현황 조회·내려받기        30         2    
3   체계 C 탄약 현황 요청(전화·메일)        45         5    
4          엑셀로 3개 자료 합치기        60         1  병목
5             품목코드 수기 대조        70         1  병목
6           부대명·날짜 형식 통일        35         1    
7       결측·이상값 확인(전화 문의)        55        10  병목
8         체계 간 불일치 원인 확인        50        12    
9        지표(재고·가동·소모) 집계        40         1    
10     군수참모 보고용 표·그래프 작성        65         2  병목
11      마스터 데이터 수정 요청·확정        45        10    
12           보고 후 수정·재배포        30         3    

업무 단계 12단계(목표 10단계 이상: 달성), 병목 4개소(목표 3개소 이상: 달성)
1회 상황 파악 AS-IS 550분 -> TO-BE 50분 (91% 단축 예상, 가정값)
```
</details>


---

## 🎯 미션 02. 서로 다른 부품코드, 부대코드, 날짜 포맷 통합 및 결측 데이터 전처리 파이프라인 구축
**소요시간**: 약 2.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

부품코드, 부대코드, 날짜 포맷 통합 및 결측 데이터 전처리 파이프라인을 구축한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **체계마다 다른 파일 형식(CSV, JSON, 탭 구분 엑셀 내보내기)과 품목코드·부대코드·날짜 형식을 표준 키로 통합하고 결측 데이터를 전처리하는 파이프라인을 구축한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **세 체계의 가상 보고를 메모리 안의 문자열로 만들어 형식별 파서로 읽고, 머리글 별칭 사전으로 양식 변경에 대응 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/connectors/dat09_data_loader.py 및 스키마 정의서` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 국방 서식/양식 변경에 대한 유연한 확장성

#### 미션 PASS 조건

- 정량 기준: 지원 파일 포맷(CSV, Excel, JSON 등) 파싱 성공률 100%


- 진행 기준: 데이터 로더 및 스키마 유효성 검사 모듈 완성


- 검증 방법: 다양한 형태의 샘플 군수 문서 대상 파싱 테스트


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""체계별로 다른 형식(CSV·JSON·탭 구분 엑셀 내보내기)의 가상 재고 보고를 메모리 안에서 파싱하고, 품목코드·부대코드·날짜 형식을 표준 키로 통합하며 결측값을 체계 간 교차 값으로 보완하는 전처리 파이프라인을 구축한다."""
import csv
import datetime
import io
import json
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 이기종 군수정보체계 3종(A 보급, B 정비, C 탄약)의 재고 보고 생성기
# 실제 체계명·부대·품목·수량 없음. 같은 (품목, 부대) 재고를 세 체계가 서로 다른 코드·날짜 형식으로 보고한다.
# 기준 날짜: 2026-09-30(가상). 마스터(정답) 재고 TRUE 와 체계별 보고에는 의도적 불일치·결측이 섞여 있다.
# ---------------------------------------------------------------------------
N_PART, N_UNIT = 60, 12
CATS = ["급식", "피복", "유류", "정비소모", "수리부속", "탄약"]
PARTS = {p: {"name": f"가상품목{p:02d}", "cat": CATS[(p - 1) % 6]} for p in range(1, N_PART + 1)}
FMT = {"A": dict(part=lambda p: f"AP-{p:05d}", unit=lambda u: f"U{u:02d}", date=lambda d: d.strftime("%Y%m%d")),
       "B": dict(part=lambda p: f"{p:05d}-X", unit=lambda u: f"가상{u:02d}대대", date=lambda d: d.strftime("%Y-%m-%d")),
       "C": dict(part=lambda p: f"P{p}", unit=lambda u: str(u), date=lambda d: d.strftime("%d/%m/%Y"))}
BASE_DATE = datetime.date(2026, 9, 30)


def make_systems(seed=9, mismatch=0.10, missing=0.03):
    """TRUE(정답 재고)와 체계별 원본 보고 레코드(체계 고유 형식)를 만든다."""
    rng = np.random.default_rng(seed)
    true = {(p, u): int(rng.integers(20, 500)) for p in PARTS for u in range(1, N_UNIT + 1)}
    out = {s: [] for s in "ABC"}
    truth_diff = {}                                   # (체계, p, u) -> 불일치 원인(정답)
    for (p, u), q in true.items():
        for s in "ABC":
            qty, cause = q, None
            r = rng.random()
            if r < mismatch / 3:                      # 입력 오류(자릿수/오타)
                qty, cause = q + int(rng.choice([-1, 1]) * rng.integers(10, 60)), "입력 오류"
            elif r < 2 * mismatch / 3:                # 반영 지연(전일 값)
                qty, cause = q + int(rng.integers(5, 40)), "반영 지연"
            elif r < mismatch:                        # 단위 오류(10배)
                qty, cause = q * 10, "단위 오류"
            d = BASE_DATE - datetime.timedelta(days=int(cause == "반영 지연"))
            if rng.random() < missing:                # 결측
                qty, cause = None, "결측"
            if cause:
                truth_diff[(s, p, u)] = cause
            out[s].append({"part": FMT[s]["part"](p), "unit": FMT[s]["unit"](u), "date": FMT[s]["date"](d), "qty": qty})
    return true, out, truth_diff


def parse_part(code):
    # [TODO : 학생 실습 2] 서로 다른 부품코드, 부대코드, 날짜 포맷 통합 및 결측 데이터 전처리 파이프라인 구축 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def parse_unit(code):
    # [TODO : 학생 실습 2] 서로 다른 부품코드, 부대코드, 날짜 포맷 통합 및 결측 데이터 전처리 파이프라인 구축 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def parse_date(s):
    if "/" in s:
        d, m, y = s.split("/"); return datetime.date(int(y), int(m), int(d))
    if "-" in s:
        return datetime.date(*map(int, s.split("-")))
    return datetime.date(int(s[:4]), int(s[4:6]), int(s[6:]))

true, raw, tdiff = make_systems()
ALIAS = {"part": ["part", "부품코드", "품목코드"], "unit": ["unit", "부대", "부대코드"], "date": ["date", "기준일", "일자"], "qty": ["qty", "수량", "재고"]}
H2F = {h: f for f, hs in ALIAS.items() for h in hs}

# --- 가상 원본 문자열 3종: 체계별 형식과 결측 표기가 다르다 ---
txt_a = "\n".join(["part,unit,date,qty"] + [f"{r['part']},{r['unit']},{r['date']},{'' if r['qty'] is None else r['qty']}" for r in raw["A"]])
txt_b = json.dumps({"system": "B", "items": [{"부품코드": r["part"], "부대": r["unit"], "기준일": r["date"], "수량": r["qty"]} for r in raw["B"]]}, ensure_ascii=False)
txt_c = "\n".join(["품목코드\t부대코드\t일자\t재고"] + [f"{r['part']}\t{r['unit']}\t{r['date']}\t{-1 if r['qty'] is None else r['qty']}" for r in raw["C"]])
TEXT = {"A": txt_a, "B": txt_b, "C": txt_c}
# 파싱 실패 유발 줄 추가(보류 대상)
TEXT["A"] += "\nAP-9999x,U01,20260930,12\nAP-00001,U99,20260930,12"

def norm(row, system):
    r = {H2F[k.strip()]: v for k, v in row.items() if k.strip() in H2F}
    q = r["qty"]
    if system == "A":
        q = None if q in ("", None) else int(q)
    elif system == "B":
        q = None if q is None else int(q)
    else:
        q = None if int(q) == -1 else int(q)
    return {"sys": system, "p": parse_part(r["part"]), "u": parse_unit(r["unit"]), "date": parse_date(r["date"]), "qty": q}

READ = {"A": lambda s: list(csv.DictReader(io.StringIO(s))), "B": lambda s: json.loads(s)["items"],
        "C": lambda s: list(csv.DictReader(io.StringIO(s), delimiter="\t"))}
ok_fmt, recs, hold = {}, [], []
for sysname, text in TEXT.items():
    try:
        rows = READ[sysname](text); ok_fmt[sysname] = True
    except Exception:
        rows = []; ok_fmt[sysname] = False
    for r in rows:
        try:
            n = norm(r, sysname)
            if not (1 <= n["p"] <= N_PART and 1 <= n["u"] <= N_UNIT):
                raise ValueError("키 범위 밖")
            recs.append(n)
        except Exception as ex:
            hold.append((sysname, str(r)[:60], str(ex)[:20]))
df = pd.DataFrame(recs)
print(f"[형식별 파싱] A(CSV) {'성공' if ok_fmt['A'] else '실패'}, B(JSON) {'성공' if ok_fmt['B'] else '실패'}, C(탭 구분) {'성공' if ok_fmt['C'] else '실패'}")
print(f"표준화 {len(df)}건(체계별 {df['sys'].value_counts().to_dict()}), 반영 보류 {len(hold)}건")
for h in hold:
    print("  보류:", h)
rate = sum(ok_fmt.values()) / len(ok_fmt)

# --- 표준 키 통합: (품목, 부대) 기준으로 세 체계를 한 행에 ---
wide = df.pivot_table(index=["p", "u"], columns="sys", values="qty", aggfunc="first")
print(f"\n통합 키 {len(wide)}개(기대 {N_PART * N_UNIT}), 결측 값 {int(wide.isna().sum().sum())}건 (A {wide['A'].isna().sum()}, B {wide['B'].isna().sum()}, C {wide['C'].isna().sum()})")
# --- 결측 전처리: 다른 두 체계 값 중 작은 값(단위 10배 오류 방지)을 후보로 보완하고 표시, 둘 다 없으면 미해결 ---
filled = wide.copy(); flag = pd.DataFrame("", index=wide.index, columns=wide.columns)
for s in "ABC":
    others = [c for c in "ABC" if c != s]
    m = wide[s].isna()
    vals = wide.loc[m, others]
    fill = vals.min(axis=1)
    filled.loc[m, s] = fill
    flag.loc[m & fill.notna(), s] = "보완"
    flag.loc[m & fill.isna(), s] = "미해결"
n_fill = int((flag == "보완").sum().sum()); n_un = int((flag == "미해결").sum().sum())
print(f"결측 보완 {n_fill}건(다른 체계 값 기반 후보값, 확정 아님), 미해결 {n_un}건")
t = true
exact = np.mean([filled.loc[(p, u), s] == t[(p, u)] for (p, u) in list(wide.index)[:720] for s in "ABC" if flag.loc[(p, u), s] == "보완"])
print(f"보완값이 마스터와 같은 비율(가상 정답 대조) {exact:.0%} -> 보완값은 '후보'로 표시하고 마스터 확정은 사람이 한다")
print("\n표준 통합표 미리보기(3행):")
print(filled.head(3).astype("float").to_string())
print(f"\n목표(3개 형식 파싱 성공률 100%, 양식 변경 대응, 결측 전처리): {'달성' if rate == 1 and len(wide) == 720 and len(hold) == 2 else '미달'}")
print("새 머리글 대응은 ALIAS 사전에 별칭만 추가하면 된다(코드 수정 불필요).")


<details>
<summary><b>💡 [참고] 미션 02 예상 실행 결과 (Output)</b></summary>

```text
[형식별 파싱] A(CSV) 성공, B(JSON) 성공, C(탭 구분) 성공
표준화 2160건(체계별 {'A': 720, 'B': 720, 'C': 720}), 반영 보류 2건
  보류: ('A', "{'part': 'AP-9999x', 'unit': 'U01', 'date': '20260930', 'qty", 'invalid literal for ')
  보류: ('A', "{'part': 'AP-00001', 'unit': 'U99', 'date': '20260930', 'qty", '키 범위 밖')

통합 키 720개(기대 720), 결측 값 46건 (A 18, B 14, C 14)
결측 보완 46건(다른 체계 값 기반 후보값, 확정 아님), 미해결 0건
보완값이 마스터와 같은 비율(가상 정답 대조) 100% -> 보완값은 '후보'로 표시하고 마스터 확정은 사람이 한다

표준 통합표 미리보기(3행):
sys      A      B      C
p u                     
1 1  222.0  222.0  222.0
  2  437.0  437.0  437.0
  3  444.0  481.0  481.0

목표(3개 형식 파싱 성공률 100%, 양식 변경 대응, 결측 전처리): 달성
새 머리글 대응은 ALIAS 사전에 별칭만 추가하면 된다(코드 수정 불필요).
```
</details>


---

## 🎯 미션 03. 통합 데이터베이스 1차 구축 및 핵심 군수 지표 집계 Baseline 쿼리 모듈 구현
**소요시간**: 약 2.5시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

통합 DB 구축 및 핵심 군수 지표 집계 Baseline 쿼리 모듈을 구현한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **세 체계의 표준화 재고를 통합 데이터 저장소에 구축하고, 검증 룰 15종과 핵심 군수 지표 집계 Baseline 쿼리를 구현한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **표준화한 2,160건을 하나의 표로 쌓아 통합 저장소로 삼고, 룰 15종을 가상 조항 번호가 붙은 설정 문자열로 분리해 위반 건수와 처리 지연시간을 측정 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/engine/dat09_rule_engine.py` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 군 규정 및 지침 조항과의 정확한 일치성

#### 미션 PASS 조건

- 정량 기준: 업무 검증 룰 15종 이상 코딩, 룰 처리 지연시간 50ms 이하


- 진행 기준: 룰 엔진 단위 테스트 100% 통과


- 검증 방법: 경계 조건(Boundary condition)에 대한 룰 판정 검증


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""세 체계의 표준화 재고를 하나의 통합 데이터 저장소(표)로 구축하고, 설정 문자열로 분리한 검증 룰 15종과 핵심 군수 지표(재고 합계·부족 품목·체계 간 일치율) 집계 Baseline 쿼리를 구현해 처리 지연시간과 경계 조건 테스트를 확인한다."""
import datetime
import time
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 이기종 군수정보체계 3종(A 보급, B 정비, C 탄약)의 재고 보고 생성기
# 실제 체계명·부대·품목·수량 없음. 같은 (품목, 부대) 재고를 세 체계가 서로 다른 코드·날짜 형식으로 보고한다.
# 기준 날짜: 2026-09-30(가상). 마스터(정답) 재고 TRUE 와 체계별 보고에는 의도적 불일치·결측이 섞여 있다.
# ---------------------------------------------------------------------------
N_PART, N_UNIT = 60, 12
CATS = ["급식", "피복", "유류", "정비소모", "수리부속", "탄약"]
PARTS = {p: {"name": f"가상품목{p:02d}", "cat": CATS[(p - 1) % 6]} for p in range(1, N_PART + 1)}
FMT = {"A": dict(part=lambda p: f"AP-{p:05d}", unit=lambda u: f"U{u:02d}", date=lambda d: d.strftime("%Y%m%d")),
       "B": dict(part=lambda p: f"{p:05d}-X", unit=lambda u: f"가상{u:02d}대대", date=lambda d: d.strftime("%Y-%m-%d")),
       "C": dict(part=lambda p: f"P{p}", unit=lambda u: str(u), date=lambda d: d.strftime("%d/%m/%Y"))}
BASE_DATE = datetime.date(2026, 9, 30)


def make_systems(seed=9, mismatch=0.10, missing=0.03):
    """TRUE(정답 재고)와 체계별 원본 보고 레코드(체계 고유 형식)를 만든다."""
    rng = np.random.default_rng(seed)
    true = {(p, u): int(rng.integers(20, 500)) for p in PARTS for u in range(1, N_UNIT + 1)}
    out = {s: [] for s in "ABC"}
    truth_diff = {}                                   # (체계, p, u) -> 불일치 원인(정답)
    for (p, u), q in true.items():
        for s in "ABC":
            qty, cause = q, None
            r = rng.random()
            if r < mismatch / 3:                      # 입력 오류(자릿수/오타)
                qty, cause = q + int(rng.choice([-1, 1]) * rng.integers(10, 60)), "입력 오류"
            elif r < 2 * mismatch / 3:                # 반영 지연(전일 값)
                qty, cause = q + int(rng.integers(5, 40)), "반영 지연"
            elif r < mismatch:                        # 단위 오류(10배)
                qty, cause = q * 10, "단위 오류"
            d = BASE_DATE - datetime.timedelta(days=int(cause == "반영 지연"))
            if rng.random() < missing:                # 결측
                qty, cause = None, "결측"
            if cause:
                truth_diff[(s, p, u)] = cause
            out[s].append({"part": FMT[s]["part"](p), "unit": FMT[s]["unit"](u), "date": FMT[s]["date"](d), "qty": qty})
    return true, out, truth_diff


def parse_part(code):
    # [TODO : 학생 실습 3] 통합 데이터베이스 1차 구축 및 핵심 군수 지표 집계 Baseline 쿼리 모듈 구현 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def parse_unit(code):
    # [TODO : 학생 실습 3] 통합 데이터베이스 1차 구축 및 핵심 군수 지표 집계 Baseline 쿼리 모듈 구현 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def parse_date(s):
    if "/" in s:
        d, m, y = s.split("/"); return datetime.date(int(y), int(m), int(d))
    if "-" in s:
        return datetime.date(*map(int, s.split("-")))
    return datetime.date(int(s[:4]), int(s[4:6]), int(s[6:]))

true, raw, tdiff = make_systems()
rows = []
for s in "ABC":
    for r in raw[s]:
        rows.append({"sys": s, "p": parse_part(r["part"]), "u": parse_unit(r["unit"]), "date": parse_date(r["date"]), "qty": r["qty"]})
store = pd.DataFrame(rows)                                   # 통합 데이터 저장소(체계 3개 x 720키)
store["cat"] = store["p"].map(lambda p: PARTS[p]["cat"])
store["qty_f"] = store["qty"].astype("float")

# 1) 검증 룰 15종(설정): (ID, 가상 조항, 조건식(참이면 정상), 메시지)
RULES = [("D01", "가상지침 1조", "1 <= p <= 60", "품목 키 범위 밖"), ("D02", "가상지침 1조", "1 <= u <= 12", "부대 키 범위 밖"),
         ("D03", "가상지침 2조", "sys in ('A', 'B', 'C')", "알 수 없는 체계"), ("D04", "가상지침 3조", "qty is not None", "수량 결측"),
         ("D05", "가상지침 3조", "qty is None or qty >= 0", "수량 음수"), ("D06", "가상지침 3조", "qty is None or qty <= 5000", "수량 상한 초과"),
         ("D07", "가상지침 4조", "isinstance(p, int) and isinstance(u, int)", "키 자료형 오류"), ("D08", "가상지침 4조", "date <= TODAY", "미래 날짜"),
         ("D09", "가상지침 5조", "(TODAY - date).days <= 3", "3일 초과 지연 보고"), ("D10", "가상지침 5조", "(TODAY - date).days >= 0", "기준일 이후 날짜"),
         ("D11", "가상지침 6조", "cat in CATS", "분류 코드 오류"), ("D12", "가상지침 6조", "qty is None or qty % 1 == 0", "수량 소수 불가"),
         ("D13", "가상지침 7조", "not (sys == 'C' and cat != '탄약' and qty is not None and qty > 3000)", "탄약체계 비탄약 대량 수량"),
         ("D14", "가상지침 7조", "qty is None or qty < 2000 or qty % 10 != 0", "10배 단위 오류 의심(2000 이상 10의 배수)"),
         ("D15", "가상지침 8조", "u != 0", "부대 0 사용 불가")]
TODAY = BASE_DATE
CODE = [(i, a, compile(e, i, "eval"), m) for i, a, e, m in RULES]
SAFE = {"__builtins__": {"isinstance": isinstance, "int": int}}


def check(rec):
    env = {"CATS": CATS, "TODAY": TODAY, **rec}
    bad = []
    for i, a, c, m in CODE:
        try:
            ok = eval(c, SAFE, env)
        except Exception:
            ok = False
        if not ok:
            bad.append((i, m))
    return bad


recs = store[["sys", "p", "u", "date", "qty", "cat"]].astype(object).where(store[["sys", "p", "u", "date", "qty", "cat"]].notna(), None).to_dict("records")
lat, viol = [], {}
for r in recs:
    t0 = time.perf_counter(); res = check(r); lat.append((time.perf_counter() - t0) * 1000)
    for i, m in res:
        viol[i] = viol.get(i, 0) + 1
lat = np.array(lat)
print(f"[검증 룰 {len(RULES)}종] 대상 {len(recs)}건, 룰 위반 {sum(viol.values())}건 [교육용가상자료]")
print("룰별 위반:", dict(sorted(viol.items())))
print(f"룰 처리 지연 평균 {lat.mean():.3f}ms, 최대 {lat.max():.3f}ms (목표 50ms 이하: {'달성' if lat.max() <= 50 else '미달'}, 환경에 따라 다름)")

# 2) Baseline 지표 쿼리: 함수로 분리(체계 A를 기준 값으로)
def q_stock_by_cat(df, sysname="A"):
    return df[df["sys"] == sysname].groupby("cat")["qty_f"].sum()


def q_low_stock(df, thr=60, sysname="A"):
    d = df[(df["sys"] == sysname) & (df["qty_f"] < thr)]
    return d.groupby("p")["u"].count().sort_values(ascending=False)


def q_agreement(df):
    w = df.pivot_table(index=["p", "u"], columns="sys", values="qty_f")
    ok = w.notna().all(axis=1)
    return float((w[ok].nunique(axis=1) == 1).mean()), int(ok.sum())


t0 = time.perf_counter()
cat_sum, low, (agree, n_cmp) = q_stock_by_cat(store), q_low_stock(store), q_agreement(store)
print("\n[Baseline 지표 쿼리] 쿼리 3개 실행", f"{(time.perf_counter() - t0) * 1000:.1f}ms")
print("분류별 재고 합계(체계 A 기준):", cat_sum.astype(int).to_dict())
print("재고 60 미만 부대가 많은 품목 상위 3:", low.head(3).to_dict())
print(f"세 체계 모두 보고한 {n_cmp}개 키 중 수량이 완전히 같은 비율(체계 간 일치율) {agree:.1%} -> 이후 미션에서 불일치 원인을 해결")

# 3) 경계 조건 단위 테스트
base = {"sys": "A", "p": 1, "u": 1, "date": TODAY, "qty": 100, "cat": "급식"}
tests = [("정상", base, []), ("수량 0 허용", {**base, "qty": 0}, []), ("수량 상한 5000 경계(D06 통과, D14 의심)", {**base, "qty": 5000}, ["D14"]),
         ("수량 5001", {**base, "qty": 5001}, ["D06"]), ("지연 3일 허용", {**base, "date": TODAY - datetime.timedelta(days=3)}, []),
         ("지연 4일", {**base, "date": TODAY - datetime.timedelta(days=4)}, ["D09"]), ("미래 날짜", {**base, "date": TODAY + datetime.timedelta(days=1)}, ["D08", "D10"]),
         ("품목 키 61", {**base, "p": 61}, ["D01"]), ("수량 결측", {**base, "qty": None}, ["D04"]), ("수량 소수", {**base, "qty": 10.5}, ["D12"])]
ok = 0
print("\n[경계 조건 단위 테스트]")
for n, r, exp in tests:
    got = sorted(i for i, _ in check(r))
    p = got == exp
    ok += p
    print(f" - {n}: 기대 {exp or '위반 없음'} / 결과 {got or '위반 없음'} -> {'PASS' if p else 'FAIL'}")
print(f"단위 테스트 {ok}/{len(tests)} 통과({ok / len(tests):.0%})")
print(f"목표(룰 15종 이상, 지연 50ms 이하, 단위 테스트 100%): {'달성' if len(RULES) >= 15 and lat.max() <= 50 and ok == len(tests) else '미달'}")


<details>
<summary><b>💡 [참고] 미션 03 예상 실행 결과 (Output)</b></summary>

```text
[검증 룰 15종] 대상 2160건, 룰 위반 107건 [교육용가상자료]
룰별 위반: {'D04': 46, 'D05': 3, 'D13': 11, 'D14': 47}
룰 처리 지연 평균 0.006ms, 최대 0.185ms (목표 50ms 이하: 달성, 환경에 따라 다름)

[Baseline 지표 쿼리] 쿼리 3개 실행 32.7ms
분류별 재고 합계(체계 A 기준): {'급식': 33087, '수리부속': 42203, '유류': 40597, '정비소모': 38378, '탄약': 39855, '피복': 39222}
재고 60 미만 부대가 많은 품목 상위 3: {2: 3, 10: 3, 60: 3}
세 체계 모두 보고한 677개 키 중 수량이 완전히 같은 비율(체계 간 일치율) 74.0% -> 이후 미션에서 불일치 원인을 해결

[경계 조건 단위 테스트]
 - 정상: 기대 위반 없음 / 결과 위반 없음 -> PASS
 - 수량 0 허용: 기대 위반 없음 / 결과 위반 없음 -> PASS
 - 수량 상한 5000 경계(D06 통과, D14 의심): 기대 ['D14'] / 결과 ['D14'] -> PASS
 - 수량 5001: 기대 ['D06'] / 결과 ['D06'] -> PASS
 - 지연 3일 허용: 기대 위반 없음 / 결과 위반 없음 -> PASS
 - 지연 4일: 기대 ['D09'] / 결과 ['D09'] -> PASS
 - 미래 날짜: 기대 ['D08', 'D10'] / 결과 ['D08', 'D10'] -> PASS
 - 품목 키 61: 기대 ['D01'] / 결과 ['D01'] -> PASS
 - 수량 결측: 기대 ['D04'] / 결과 ['D04'] -> PASS
 - 수량 소수: 기대 ['D12'] / 결과 ['D12'] -> PASS
단위 테스트 10/10 통과(100%)
목표(룰 15종 이상, 지연 50ms 이하, 단위 테스트 100%): 달성
```
</details>


---

## 🎯 미션 04. 실시간 데이터 스트리밍(Kafka/CDC) 기반 이기종 데이터 정합성 유지 및 자동 동기화 고도화
**소요시간**: 약 3.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

실시간 데이터 스트리밍 기반 이기종 데이터 정합성 유지 및 자동 동기화를 고도화한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **변경 이벤트 스트림(CDC 방식을 메모리에서 모의)으로 체계 간 정합성을 유지하고 불일치 원인을 분류해 마스터 값 후보를 추천하는 모듈을 고도화한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **외부 스트리밍 도구 없이 메모리 안의 큐로 변경 이벤트를 만들고, 중복 전송과 순서가 뒤바뀐 오래된 값이 섞인 상태에서 이벤트 ID 중복 제거와 키별 최신 시각 우선 규칙으로 동기화 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/ai/dat09_intelligence_module.py` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 추천 사유의 합리성 및 실무 수용성

#### 미션 PASS 조건

- 정량 기준: 추천/분류 정확도 85% 이상 달성


- 진행 기준: AI 추론 모듈 구현 및 벤치마크 테스트 완료


- 검증 방법: 과거 승인/처리 완료 데이터와의 추천 일치도 검증


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""변경 이벤트 스트림(CDC 방식 모의)을 순서 뒤바뀜·중복 이벤트까지 처리해 통합 저장소에 자동 동기화하고, 체계 간 불일치의 원인을 분류해 마스터 값 후보를 추천하는 모듈을 구현해 분류·추천 정확도를 측정한다."""
import collections
import datetime
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 이기종 군수정보체계 3종(A 보급, B 정비, C 탄약)의 재고 보고 생성기
# 실제 체계명·부대·품목·수량 없음. 같은 (품목, 부대) 재고를 세 체계가 서로 다른 코드·날짜 형식으로 보고한다.
# 기준 날짜: 2026-09-30(가상). 마스터(정답) 재고 TRUE 와 체계별 보고에는 의도적 불일치·결측이 섞여 있다.
# ---------------------------------------------------------------------------
N_PART, N_UNIT = 60, 12
CATS = ["급식", "피복", "유류", "정비소모", "수리부속", "탄약"]
PARTS = {p: {"name": f"가상품목{p:02d}", "cat": CATS[(p - 1) % 6]} for p in range(1, N_PART + 1)}
FMT = {"A": dict(part=lambda p: f"AP-{p:05d}", unit=lambda u: f"U{u:02d}", date=lambda d: d.strftime("%Y%m%d")),
       "B": dict(part=lambda p: f"{p:05d}-X", unit=lambda u: f"가상{u:02d}대대", date=lambda d: d.strftime("%Y-%m-%d")),
       "C": dict(part=lambda p: f"P{p}", unit=lambda u: str(u), date=lambda d: d.strftime("%d/%m/%Y"))}
BASE_DATE = datetime.date(2026, 9, 30)


def make_systems(seed=9, mismatch=0.10, missing=0.03):
    """TRUE(정답 재고)와 체계별 원본 보고 레코드(체계 고유 형식)를 만든다."""
    rng = np.random.default_rng(seed)
    true = {(p, u): int(rng.integers(20, 500)) for p in PARTS for u in range(1, N_UNIT + 1)}
    out = {s: [] for s in "ABC"}
    truth_diff = {}                                   # (체계, p, u) -> 불일치 원인(정답)
    for (p, u), q in true.items():
        for s in "ABC":
            qty, cause = q, None
            r = rng.random()
            if r < mismatch / 3:                      # 입력 오류(자릿수/오타)
                qty, cause = q + int(rng.choice([-1, 1]) * rng.integers(10, 60)), "입력 오류"
            elif r < 2 * mismatch / 3:                # 반영 지연(전일 값)
                qty, cause = q + int(rng.integers(5, 40)), "반영 지연"
            elif r < mismatch:                        # 단위 오류(10배)
                qty, cause = q * 10, "단위 오류"
            d = BASE_DATE - datetime.timedelta(days=int(cause == "반영 지연"))
            if rng.random() < missing:                # 결측
                qty, cause = None, "결측"
            if cause:
                truth_diff[(s, p, u)] = cause
            out[s].append({"part": FMT[s]["part"](p), "unit": FMT[s]["unit"](u), "date": FMT[s]["date"](d), "qty": qty})
    return true, out, truth_diff


def parse_part(code):
    # [TODO : 학생 실습 4] 실시간 데이터 스트리밍(Kafka/CDC) 기반 이기종 데이터 정합성 유지 및 자동 동기화 고도화 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def parse_unit(code):
    # [TODO : 학생 실습 4] 실시간 데이터 스트리밍(Kafka/CDC) 기반 이기종 데이터 정합성 유지 및 자동 동기화 고도화 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def parse_date(s):
    if "/" in s:
        d, m, y = s.split("/"); return datetime.date(int(y), int(m), int(d))
    if "-" in s:
        return datetime.date(*map(int, s.split("-")))
    return datetime.date(int(s[:4]), int(s[4:6]), int(s[6:]))

true, raw, tdiff = make_systems()
rng = np.random.default_rng(4)

# 1) 이벤트 스트림 생성: 각 체계 보고를 (체계, 키, 값, 시각, 이벤트ID) 이벤트로 바꾸고 중복·순서 뒤바뀜을 주입
events = []
for s in "ABC":
    for i, r in enumerate(raw[s]):
        events.append({"id": f"{s}{i:04d}", "sys": s, "p": parse_part(r["part"]), "u": parse_unit(r["unit"]), "qty": r["qty"],
                       "ts": int(rng.integers(0, 10000)), "day": parse_date(r["date"])})
dups = [dict(events[i]) for i in rng.choice(len(events), 120, replace=False)]            # 중복 전송 120건
stale = []                                                                               # 같은 키의 오래된 값(늦게 도착한 옛 이벤트) 80건
for i in rng.choice(len(events), 80, replace=False):
    e = dict(events[i]); e["id"] += "-old"; e["ts"] = e["ts"] - 5000; e["qty"] = (e["qty"] or 0) + 77
    stale.append(e)
stream = events + dups + stale
rng.shuffle(stream)                                                                      # 도착 순서 뒤섞기
print(f"[스트림] 이벤트 {len(stream)}건 (정상 {len(events)}, 중복 {len(dups)}, 오래된 값 {len(stale)}), 도착 순서 무작위")

# 2) 자동 동기화: 이벤트 ID 중복 제거 + 키별 최신 시각 우선(last-write-wins)
store, seen, applied, skipped_dup, skipped_old = {}, set(), 0, 0, 0
queue = collections.deque(stream)
while queue:
    e = queue.popleft()
    if e["id"] in seen:
        skipped_dup += 1; continue
    seen.add(e["id"])
    k = (e["sys"], e["p"], e["u"])
    if k in store and store[k]["ts"] >= e["ts"]:
        skipped_old += 1; continue
    store[k] = {"qty": e["qty"], "ts": e["ts"], "day": e["day"]}; applied += 1
exp = {(e["sys"], e["p"], e["u"]): e["qty"] for e in events}
match = sum(store[k]["qty"] == v for k, v in exp.items())
print(f"동기화: 적용 {applied}건, 중복 건너뜀 {skipped_dup}건, 오래된 값 건너뜀 {skipped_old}건")
print(f"일괄 스냅샷 대비 동기화 결과 일치 {match}/{len(exp)}건({match / len(exp):.1%}) -> 스트림 방식과 일괄 방식의 결과가 같아야 한다")

# 3) 불일치 감지·원인 분류·마스터 후보 추천
rows = []
for p in PARTS:
    for u in range(1, N_UNIT + 1):
        vals = {s: store[(s, p, u)] for s in "ABC"}
        q = {s: v["qty"] for s, v in vals.items()}
        have = {s: x for s, x in q.items() if x is not None}
        cnt = collections.Counter(have.values())
        top, n_top = cnt.most_common(1)[0] if cnt else (None, 0)
        for s in "ABC":
            if q[s] == top and n_top >= 2:
                continue
            if q[s] is None:
                cause = "결측"
            elif n_top >= 2 and q[s] != top:
                lag = (BASE_DATE - vals[s]["day"]).days
                cause = "단위 오류" if q[s] == top * 10 else ("반영 지연" if lag >= 1 else "입력 오류")
            else:
                cause = "판단 불가"                       # 세 값이 모두 다르면 사람이 판단
            rows.append({"sys": s, "p": p, "u": u, "cause": cause, "recommend": top if n_top >= 2 else None, "true": true[(p, u)]})
d = pd.DataFrame(rows)
d["truth_cause"] = [tdiff.get((r.sys, r.p, r.u)) for r in d.itertuples()]
auto = d[d["cause"] != "판단 불가"]
cause_acc = float((auto["cause"] == auto["truth_cause"]).mean())
rec_ok = float((auto["recommend"] == auto["true"]).mean())
true_cells = pd.Series(tdiff)
print(f"\n[불일치 분류] 불일치 셀 {len(d)}건 (정답상 불일치 {len(true_cells)}건), 자동 분류 {len(auto)}건, 사람 판단 필요 {len(d) - len(auto)}건")
print(pd.crosstab(auto["truth_cause"].fillna("정상(오탐)"), auto["cause"]).to_string())
print(f"원인 분류 정확도 {cause_acc:.1%}, 마스터 값 후보(다수결) 정확도 {rec_ok:.1%}, 감지 재현율 {len(set(zip(d.sys, d.p, d.u)) & set(true_cells.index)) / len(true_cells):.1%}")
print("추천 사유 예시:")
for r in auto.head(3).itertuples():
    print(f"  - 체계 {r.sys} 품목 {r.p} 부대 {r.u}: 원인 '{r.cause}', 다른 두 체계 다수값 {r.recommend}을(를) 마스터 후보로 제시")
print("추천은 후보이며 마스터 데이터 확정은 데이터관리관 승인(미션 7)으로만 이루어진다.")
print(f"\n목표(추천/분류 정확도 85% 이상): 분류 {'달성' if cause_acc >= 0.85 else '미달'}, 마스터 후보 {'달성' if rec_ok >= 0.85 else '미달'}")


<details>
<summary><b>💡 [참고] 미션 04 예상 실행 결과 (Output)</b></summary>

```text
[스트림] 이벤트 2360건 (정상 2160, 중복 120, 오래된 값 80), 도착 순서 무작위
동기화: 적용 2200건, 중복 건너뜀 120건, 오래된 값 건너뜀 40건
일괄 스냅샷 대비 동기화 결과 일치 2160/2160건(100.0%) -> 스트림 방식과 일괄 방식의 결과가 같아야 한다

[불일치 분류] 불일치 셀 261건 (정답상 불일치 243건), 자동 분류 207건, 사람 판단 필요 54건
cause        결측  단위 오류  반영 지연  입력 오류
truth_cause                         
결측           46      0      0      0
단위 오류         0     57      0      0
반영 지연         0      0     53      0
입력 오류         0      0      0     49
정상(오탐)        0      0      0      2
원인 분류 정확도 99.0%, 마스터 값 후보(다수결) 정확도 94.7%, 감지 재현율 98.4%
추천 사유 예시:
  - 체계 A 품목 1 부대 3: 원인 '입력 오류', 다른 두 체계 다수값 481.0을(를) 마스터 후보로 제시
  - 체계 B 품목 1 부대 8: 원인 '반영 지연', 다른 두 체계 다수값 393.0을(를) 마스터 후보로 제시
  - 체계 C 품목 1 부대 10: 원인 '반영 지연', 다른 두 체계 다수값 363.0을(를) 마스터 후보로 제시
추천은 후보이며 마스터 데이터 확정은 데이터관리관 승인(미션 7)으로만 이루어진다.

목표(추천/분류 정확도 85% 이상): 분류 달성, 마스터 후보 달성
```
</details>


---

## 🎯 미션 05. 체계 간 불일치 데이터 자동 식별 및 데이터 신뢰도 품질 지수 산출 엔진 개발
**소요시간**: 약 3.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

체계 간 불일치 데이터 자동 식별 및 품질 신뢰도 지수 산출 엔진을 개발한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **체계 간 불일치 데이터를 자동 식별하고 데이터 품질 지수를 산출하는 엔진을 입력-분석-검토-결과 4개 화면으로 구현한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **키별로 세 체계의 값을 비교해 값 불일치와 결측을 식별하고 편차가 큰 건을 우선 검토 대상으로 표시 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/ui/dat09_app.py` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 군 실무자 친화적 디자인 및 조작 편의성

#### 미션 PASS 조건

- 정량 기준: 핵심 화면 4종(입력-분석-검토-결과) 구현 완료


- 진행 기준: UI 프로토타입 로컬 서버 구동 및 전체 페이지 네비게이션 확인


- 검증 방법: 사용성 평가(UX Walkthrough) 및 인터랙션 테스트


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""세 체계 간 불일치 데이터를 자동 식별하고 완전성·일치성·적시성·유효성 4개 차원의 데이터 품질 지수를 부대별·체계별로 산출하며, 입력-분석-검토-결과 4개 화면을 텍스트 화면으로 구현한다."""
import datetime
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 이기종 군수정보체계 3종(A 보급, B 정비, C 탄약)의 재고 보고 생성기
# 실제 체계명·부대·품목·수량 없음. 같은 (품목, 부대) 재고를 세 체계가 서로 다른 코드·날짜 형식으로 보고한다.
# 기준 날짜: 2026-09-30(가상). 마스터(정답) 재고 TRUE 와 체계별 보고에는 의도적 불일치·결측이 섞여 있다.
# ---------------------------------------------------------------------------
N_PART, N_UNIT = 60, 12
CATS = ["급식", "피복", "유류", "정비소모", "수리부속", "탄약"]
PARTS = {p: {"name": f"가상품목{p:02d}", "cat": CATS[(p - 1) % 6]} for p in range(1, N_PART + 1)}
FMT = {"A": dict(part=lambda p: f"AP-{p:05d}", unit=lambda u: f"U{u:02d}", date=lambda d: d.strftime("%Y%m%d")),
       "B": dict(part=lambda p: f"{p:05d}-X", unit=lambda u: f"가상{u:02d}대대", date=lambda d: d.strftime("%Y-%m-%d")),
       "C": dict(part=lambda p: f"P{p}", unit=lambda u: str(u), date=lambda d: d.strftime("%d/%m/%Y"))}
BASE_DATE = datetime.date(2026, 9, 30)


def make_systems(seed=9, mismatch=0.10, missing=0.03):
    """TRUE(정답 재고)와 체계별 원본 보고 레코드(체계 고유 형식)를 만든다."""
    rng = np.random.default_rng(seed)
    true = {(p, u): int(rng.integers(20, 500)) for p in PARTS for u in range(1, N_UNIT + 1)}
    out = {s: [] for s in "ABC"}
    truth_diff = {}                                   # (체계, p, u) -> 불일치 원인(정답)
    for (p, u), q in true.items():
        for s in "ABC":
            qty, cause = q, None
            r = rng.random()
            if r < mismatch / 3:                      # 입력 오류(자릿수/오타)
                qty, cause = q + int(rng.choice([-1, 1]) * rng.integers(10, 60)), "입력 오류"
            elif r < 2 * mismatch / 3:                # 반영 지연(전일 값)
                qty, cause = q + int(rng.integers(5, 40)), "반영 지연"
            elif r < mismatch:                        # 단위 오류(10배)
                qty, cause = q * 10, "단위 오류"
            d = BASE_DATE - datetime.timedelta(days=int(cause == "반영 지연"))
            if rng.random() < missing:                # 결측
                qty, cause = None, "결측"
            if cause:
                truth_diff[(s, p, u)] = cause
            out[s].append({"part": FMT[s]["part"](p), "unit": FMT[s]["unit"](u), "date": FMT[s]["date"](d), "qty": qty})
    return true, out, truth_diff


def parse_part(code):
    # [TODO : 학생 실습 5] 체계 간 불일치 데이터 자동 식별 및 데이터 신뢰도 품질 지수 산출 엔진 개발 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def parse_unit(code):
    # [TODO : 학생 실습 5] 체계 간 불일치 데이터 자동 식별 및 데이터 신뢰도 품질 지수 산출 엔진 개발 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def parse_date(s):
    if "/" in s:
        d, m, y = s.split("/"); return datetime.date(int(y), int(m), int(d))
    if "-" in s:
        return datetime.date(*map(int, s.split("-")))
    return datetime.date(int(s[:4]), int(s[4:6]), int(s[6:]))

# ---------- 화면 1: 입력 ----------
true, raw, tdiff = make_systems()
rows = [{"sys": s, "p": parse_part(r["part"]), "u": parse_unit(r["unit"]), "date": parse_date(r["date"]), "qty": r["qty"]} for s in "ABC" for r in raw[s]]
df = pd.DataFrame(rows)
df["qty_f"] = df["qty"].astype("float")
print(f"[화면 1. 입력] 3개 체계 보고 {len(df)}건(체계별 {df['sys'].value_counts().sort_index().to_dict()}) 적재, 표준 키(품목, 부대)로 통합 [교육용가상자료]")

# ---------- 화면 2: 분석(불일치 식별) ----------
w = df.pivot_table(index=["p", "u"], columns="sys", values="qty_f")
w["n"] = w[["A", "B", "C"]].notna().sum(axis=1)
w["spread"] = w[["A", "B", "C"]].max(axis=1) - w[["A", "B", "C"]].min(axis=1)
w["consensus"] = w[["A", "B", "C"]].median(axis=1)
w["mismatch"] = (w["spread"] > 0) | (w["n"] < 3)                                      # 값이 다르거나 어느 체계에 값이 없으면 불일치로 본다
print(f"[화면 2. 분석] 키 {len(w)}개 중 불일치 {int(w['mismatch'].sum())}개({w['mismatch'].mean():.1%}): 값 불일치 {int((w['spread'] > 0).sum())}개, 결측 포함 {int((w['n'] < 3).sum())}개")
big = w[w["spread"] > 0.2 * w["consensus"]]
print(f"  편차 20% 초과(우선 검토 대상) {len(big)}개")

# ---------- 품질 지수 ----------
def dims(g):
    comp = g["qty_f"].notna().mean()                                                  # 완전성: 값이 있는 비율
    key = g.set_index(["p", "u"]).index
    cons = (~w.loc[key, "mismatch"]).mean() if len(g) else 0                           # 일치성: 체계 간 같은 값인 비율
    tim = ((BASE_DATE - g["date"]).map(lambda d: d.days) <= 0).mean()                  # 적시성: 기준일 당일 보고 비율
    val = ((g["qty_f"].isna()) | ((g["qty_f"] >= 0) & (g["qty_f"] <= 2000))).mean()    # 유효성: 허용 범위(0~2000) 안의 비율
    return pd.Series({"완전성": comp, "일치성": cons, "적시성": tim, "유효성": val})


WT = {"완전성": 0.25, "일치성": 0.4, "적시성": 0.15, "유효성": 0.2}                        # 일치성 비중을 가장 높게
def index(g):
    d = dims(g)
    return pd.concat([d, pd.Series({"품질지수": 100 * sum(d[k] * v for k, v in WT.items())})])

by_sys = pd.DataFrame({k: index(g) for k, g in df.groupby("sys")}).T
by_unit = pd.DataFrame({k: index(g) for k, g in df.groupby("u")}).T

# ---------- 화면 3: 검토 ----------
print("\n[화면 3. 검토] 체계별 품질 지수(0~100)")
print(by_sys.round(3).to_string())
low = by_unit.sort_values("품질지수").head(4)
print("품질 지수가 낮은 부대 4곳(우선 점검):")
print(low.round(3).to_string())
print("검토자는 불일치 키마다 '마스터 후보 승인 요청 / 정상 확인'을 선택하며, 수정은 체계 담당자가 한다.")

# ---------- 화면 4: 결과 ----------
total = float(index(df)["품질지수"])
grade = "양호" if total >= 90 else "보완 필요" if total >= 75 else "미흡"
gt_keys = {(p, u) for (_, p, u) in tdiff}
found = {k for k, m in w["mismatch"].items() if m}
recall = len(found & gt_keys) / len(gt_keys)
print(f"\n[화면 4. 결과] 통합 데이터 품질 지수 {total:.1f}점({grade}), 불일치 식별 재현율(가상 정답 대비) {recall:.1%}")
print(f"  차원별 가중치: {WT}")
ok = recall >= 0.95
print(f"목표(핵심 화면 4종 입력-분석-검토-결과 구현, 불일치 자동 식별·품질 지수 산출): {'달성' if ok else '미달'}")


<details>
<summary><b>💡 [참고] 미션 05 예상 실행 결과 (Output)</b></summary>

```text
[화면 1. 입력] 3개 체계 보고 2160건(체계별 {'A': 720, 'B': 720, 'C': 720}) 적재, 표준 키(품목, 부대)로 통합 [교육용가상자료]
[화면 2. 분석] 키 720개 중 불일치 219개(30.4%): 값 불일치 179개, 결측 포함 43개
  편차 20% 초과(우선 검토 대상) 95개

[화면 3. 검토] 체계별 품질 지수(0~100)
     완전성    일치성    적시성    유효성    품질지수
A  0.975  0.696  0.962  0.978  86.201
B  0.981  0.696  0.969  0.979  86.472
C  0.981  0.696  0.967  0.974  86.319
품질 지수가 낮은 부대 4곳(우선 점검):
      완전성    일치성    적시성    유효성    품질지수
3   0.972  0.583  0.950  0.983  81.556
5   0.972  0.633  0.983  0.983  84.056
10  0.978  0.650  0.967  0.961  84.167
6   0.978  0.667  0.956  0.972  84.889
검토자는 불일치 키마다 '마스터 후보 승인 요청 / 정상 확인'을 선택하며, 수정은 체계 담당자가 한다.

[화면 4. 결과] 통합 데이터 품질 지수 86.3점(보완 필요), 불일치 식별 재현율(가상 정답 대비) 100.0%
  차원별 가중치: {'완전성': 0.25, '일치성': 0.4, '적시성': 0.15, '유효성': 0.2}
목표(핵심 화면 4종 입력-분석-검토-결과 구현, 불일치 자동 식별·품질 지수 산출): 달성
```
</details>


---

## 🎯 미션 06. 지휘관 및 군수참모용 통합 군수 상황판(Logistics COP) 실시간 대시보드 구현
**소요시간**: 약 2.5시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

지휘관 및 군수참모용 통합 군수 상황판(Logistics COP) 실시간 대시보드를 구현한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **지휘관과 군수참모가 보는 통합 군수 상황판(Logistics COP)을 시간별 갱신 방식으로 구현하고, 예외 상황에서도 중단되지 않는 에러 처리를 갖춘다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **변경 이벤트를 1시간 단위로 반영해 총재고, 분류별 재고, 부족 경보를 갱신하고 지휘관용 요약 화면과 군수참모용 상세 화면을 구분해 보여 준다 과정을 진행하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/utils/dat09_validator.py 및 에러 핸들러` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 사용자에게 알기 쉬운 에러 메시지 및 조치 가이드 제공

#### 미션 PASS 조건

- 정량 기준: 예외 상황 발생 시 크래시(Crash) 0건, 에러 로그 기록률 100%


- 진행 기준: 예외 주입 테스트 10종 통과 및 감사 로그 파일 생성


- 검증 방법: 비정상 데이터(결측, 특수문자, 대용량) 스트레스 테스트


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""변경 이벤트를 1시간 단위로 받아 갱신되는 통합 군수 상황판(재고·부족 경보·데이터 신뢰도)을 지휘관용·군수참모용 역할별 화면과 그래프로 구현하고, 입력 검증기와 에러 핸들러로 예외 10종을 크래시 없이 기록·안내한다."""
import datetime
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 이기종 군수정보체계 3종(A 보급, B 정비, C 탄약)의 재고 보고 생성기
# 실제 체계명·부대·품목·수량 없음. 같은 (품목, 부대) 재고를 세 체계가 서로 다른 코드·날짜 형식으로 보고한다.
# 기준 날짜: 2026-09-30(가상). 마스터(정답) 재고 TRUE 와 체계별 보고에는 의도적 불일치·결측이 섞여 있다.
# ---------------------------------------------------------------------------
N_PART, N_UNIT = 60, 12
CATS = ["급식", "피복", "유류", "정비소모", "수리부속", "탄약"]
PARTS = {p: {"name": f"가상품목{p:02d}", "cat": CATS[(p - 1) % 6]} for p in range(1, N_PART + 1)}
FMT = {"A": dict(part=lambda p: f"AP-{p:05d}", unit=lambda u: f"U{u:02d}", date=lambda d: d.strftime("%Y%m%d")),
       "B": dict(part=lambda p: f"{p:05d}-X", unit=lambda u: f"가상{u:02d}대대", date=lambda d: d.strftime("%Y-%m-%d")),
       "C": dict(part=lambda p: f"P{p}", unit=lambda u: str(u), date=lambda d: d.strftime("%d/%m/%Y"))}
BASE_DATE = datetime.date(2026, 9, 30)


def make_systems(seed=9, mismatch=0.10, missing=0.03):
    """TRUE(정답 재고)와 체계별 원본 보고 레코드(체계 고유 형식)를 만든다."""
    rng = np.random.default_rng(seed)
    true = {(p, u): int(rng.integers(20, 500)) for p in PARTS for u in range(1, N_UNIT + 1)}
    out = {s: [] for s in "ABC"}
    truth_diff = {}                                   # (체계, p, u) -> 불일치 원인(정답)
    for (p, u), q in true.items():
        for s in "ABC":
            qty, cause = q, None
            r = rng.random()
            if r < mismatch / 3:                      # 입력 오류(자릿수/오타)
                qty, cause = q + int(rng.choice([-1, 1]) * rng.integers(10, 60)), "입력 오류"
            elif r < 2 * mismatch / 3:                # 반영 지연(전일 값)
                qty, cause = q + int(rng.integers(5, 40)), "반영 지연"
            elif r < mismatch:                        # 단위 오류(10배)
                qty, cause = q * 10, "단위 오류"
            d = BASE_DATE - datetime.timedelta(days=int(cause == "반영 지연"))
            if rng.random() < missing:                # 결측
                qty, cause = None, "결측"
            if cause:
                truth_diff[(s, p, u)] = cause
            out[s].append({"part": FMT[s]["part"](p), "unit": FMT[s]["unit"](u), "date": FMT[s]["date"](d), "qty": qty})
    return true, out, truth_diff


def parse_part(code):
    # [TODO : 학생 실습 6] 지휘관 및 군수참모용 통합 군수 상황판(Logistics COP) 실시간 대시보드 구현 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def parse_unit(code):
    # [TODO : 학생 실습 6] 지휘관 및 군수참모용 통합 군수 상황판(Logistics COP) 실시간 대시보드 구현 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def parse_date(s):
    if "/" in s:
        d, m, y = s.split("/"); return datetime.date(int(y), int(m), int(d))
    if "-" in s:
        return datetime.date(*map(int, s.split("-")))
    return datetime.date(int(s[:4]), int(s[4:6]), int(s[6:]))

true, raw, tdiff = make_systems()
rng = np.random.default_rng(6)
STATE = {(p, u): true[(p, u)] for p in PARTS for u in range(1, N_UNIT + 1)}       # 상황판의 현재 재고(체계 A 기준 시작)
SEEN, LOG, HIST = set(), [], []
LOW = 40                                                                          # 부족 경보 기준(가상)


class AppError(Exception):
    def __init__(self, code, msg, guide):
        super().__init__(msg); self.code, self.msg, self.guide = code, msg, guide


def validate(ev):
    for k in ("id", "sys", "p", "u", "delta"):
        if k not in ev or ev[k] is None:
            raise AppError("E01", f"필수 항목 '{k}' 누락", f"이벤트에 '{k}' 값을 넣어 다시 전송")
    if ev["id"] in SEEN:
        raise AppError("E02", f"중복 이벤트 {ev['id']}", "이미 반영된 이벤트이므로 무시해도 됨")
    if ev["sys"] not in ("A", "B", "C"):
        raise AppError("E03", f"알 수 없는 체계 '{ev['sys']}'", "연동 체계(A·B·C)만 허용")
    try:
        p, u, dl = int(ev["p"]), int(ev["u"]), float(ev["delta"])
    except (TypeError, ValueError):
        raise AppError("E04", "숫자 형식 오류", "품목·부대·증감량은 숫자만 입력")
    if not np.isfinite(dl):
        raise AppError("E05", "증감량이 NaN 또는 무한대", "정상 숫자로 수정")
    if (p, u) not in STATE:
        raise AppError("E06", f"존재하지 않는 키(품목 {p}, 부대 {u})", "코드 변환표에서 키 확인")
    if abs(dl) > 5000:
        raise AppError("E09", f"증감량 {dl:+.0f}이 허용 범위(±5000) 초과", "자릿수 오류 여부를 출처 체계에 확인")
    if STATE[(p, u)] + dl < 0:
        raise AppError("E07", f"재고가 음수가 됨({STATE[(p, u)]} + {dl:+.0f})", "출처 체계에 증감량 재확인 요청")
    return p, u, dl


def apply(ev):
    p, u, dl = validate(ev)
    STATE[(p, u)] += int(dl); SEEN.add(ev["id"])


def safe(fn, *a):
    """에러 핸들러: 모든 예외를 로그로 남기고 사용자에게 원인과 조치를 안내한다(크래시 없음)."""
    try:
        r = fn(*a); LOG.append({"level": "INFO", "code": "OK"}); return r
    except AppError as e:
        LOG.append({"level": "ERROR", "code": e.code, "msg": e.msg, "guide": e.guide})
    except Exception as e:
        LOG.append({"level": "ERROR", "code": "E99", "msg": repr(e)[:50], "guide": "운영 담당자에게 로그 전달"})


def view(role):
    """역할별 화면: 지휘관은 요약, 군수참모는 분류·부대 상세."""
    if role not in ("지휘관", "군수참모"):
        raise AppError("E08", f"권한 없는 역할 '{role}'", "지휘관 또는 군수참모 계정으로 접속")
    s = pd.Series(STATE)
    total, low = int(s.sum()), int((s < LOW).sum())
    if role == "지휘관":
        return f"총재고 {total}, 부족 경보 {low}건"
    cat = {c: int(sum(v for (p, u), v in STATE.items() if PARTS[p]["cat"] == c)) for c in CATS}
    worst = s.groupby(level=1).apply(lambda g: int((g < LOW).sum())).sort_values(ascending=False).head(3)
    return f"총재고 {total}, 분류별 {cat}, 부족 경보 {low}건, 경보 많은 부대 {worst.to_dict()}"


# 1) 정상 운영: 6시간 동안 시간당 이벤트 80건(불출 -, 보급 +) 반영
print("[통합 군수 상황판 - 시간별 갱신] [교육용가상자료]")
for hour in range(6):
    for i in range(80):
        p, u = int(rng.integers(1, N_PART + 1)), int(rng.integers(1, N_UNIT + 1))
        d = int(rng.integers(-30, 25))
        safe(apply, {"id": f"EV{hour}-{i}", "sys": "ABC"[i % 3], "p": p, "u": u, "delta": d})
    s = pd.Series(STATE)
    cat_tot = {c: sum(v for (p, u), v in STATE.items() if PARTS[p]["cat"] == c) for c in CATS}
    HIST.append({"hour": hour + 8, "total": int(s.sum()), "low": int((s < LOW).sum()), **cat_tot})
    print(f" {hour + 8:02d}:00  지휘관 -> {view('지휘관')}")
rej = [l for l in LOG if l["level"] == "ERROR"]
print(f"정상 운영 중 거부된 이벤트 {len(rej)}건(재고 음수 보호 등):", sorted({l['code'] for l in rej}))
print("군수참모 화면:", view("군수참모"))

# 2) 예외 주입 10종
LOG.clear()
base = {"id": "T-OK", "sys": "A", "p": 1, "u": 1, "delta": 1}
safe(apply, base)                                  # 먼저 정상 반영해 두어야 같은 번호 재전송이 중복이 된다(시험 대상 아님)
LOG.clear()
cases = [("필수 항목 누락", {"id": "T1", "sys": "A", "p": 1, "u": 1}), ("중복 이벤트", base),
         ("알 수 없는 체계", {**base, "id": "T3", "sys": "Z"}), ("숫자 아님", {**base, "id": "T4", "delta": "10개"}),
         ("NaN 값", {**base, "id": "T5", "delta": float("nan")}), ("없는 키", {**base, "id": "T6", "p": 999}),
         ("증감량 과대", {**base, "id": "T7", "delta": -99999}), ("재고 음수화", {**base, "id": "T8", "delta": -4000}),
         ("권한 없는 역할", "병사"), ("빈 이벤트", {})]
crash = 0
for n, c in cases:
    try:
        safe(view, c) if isinstance(c, str) else safe(apply, c)
    except BaseException:
        crash += 1
errs = [l for l in LOG if l["level"] == "ERROR"]
print("\n[예외 주입 결과 - 사용자 안내 메시지]")
for (n, c), l in zip(cases, errs):
    print(f" {l['code']} {n:<14} {l['msg']:<32} -> 조치: {l['guide']}")
exp_errors = len(cases)
rate = len(errs) / exp_errors
print(f"예외 {exp_errors}종 중 로그 기록 {len(errs)}건(기록률 {rate:.0%}), 크래시 {crash}건")
print("로그 예시:", json.dumps(errs[0], ensure_ascii=False))
print(f"목표(크래시 0건, 에러 로그 기록률 100%): {'달성' if crash == 0 and rate == 1 else '미달'}")

h = pd.DataFrame(HIST)
fig, ax = plt.subplots(2, 2, figsize=(11, 7))
for c in CATS:
    ax[0, 0].plot(h["hour"], h[c], marker="o", label=f"cat{CATS.index(c) + 1}")
ax[0, 0].legend(fontsize=7, ncol=3); ax[0, 0].set_title("Stock by category (hourly)")
ax[0, 1].plot(h["hour"], h["low"], color="tab:red", marker="s"); ax[0, 1].set_title("Shortage alerts (stock < 40)")
s = pd.Series(STATE)
ax[1, 0].bar(range(1, N_UNIT + 1), s.groupby(level=1).sum().values, color="tab:blue"); ax[1, 0].set_title("Total stock by unit (latest)")
ax[1, 1].hist(s.values, bins=20, color="tab:gray"); ax[1, 1].axvline(LOW, color="red", ls="--"); ax[1, 1].set_title("Stock distribution (red = alert line)")
plt.suptitle("DAT09 logistics COP (synthetic)"); plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 06 예상 실행 결과 (Output)</b></summary>

```text
[통합 군수 상황판 - 시간별 갱신] [교육용가상자료]
 08:00  지휘관 -> 총재고 188532, 부족 경보 27건
 09:00  지휘관 -> 총재고 188232, 부족 경보 28건
 10:00  지휘관 -> 총재고 188292, 부족 경보 26건
 11:00  지휘관 -> 총재고 187942, 부족 경보 29건
 12:00  지휘관 -> 총재고 187663, 부족 경보 27건
 13:00  지휘관 -> 총재고 187444, 부족 경보 28건
정상 운영 중 거부된 이벤트 0건(재고 음수 보호 등): []
군수참모 화면: 총재고 187444, 분류별 {'급식': 30500, '피복': 34149, '유류': 30600, '정비소모': 30650, '수리부속': 31493, '탄약': 30052}, 부족 경보 28건, 경보 많은 부대 {1: 4, 3: 4, 8: 4}

[예외 주입 결과 - 사용자 안내 메시지]
 E01 필수 항목 누락       필수 항목 'delta' 누락                 -> 조치: 이벤트에 'delta' 값을 넣어 다시 전송
 E02 중복 이벤트         중복 이벤트 T-OK                      -> 조치: 이미 반영된 이벤트이므로 무시해도 됨
 E03 알 수 없는 체계      알 수 없는 체계 'Z'                    -> 조치: 연동 체계(A·B·C)만 허용
 E04 숫자 아님          숫자 형식 오류                         -> 조치: 품목·부대·증감량은 숫자만 입력
 E05 NaN 값          증감량이 NaN 또는 무한대                  -> 조치: 정상 숫자로 수정
 E06 없는 키           존재하지 않는 키(품목 999, 부대 1)          -> 조치: 코드 변환표에서 키 확인
 E09 증감량 과대         증감량 -99999이 허용 범위(±5000) 초과      -> 조치: 자릿수 오류 여부를 출처 체계에 확인
 E07 재고 음수화         재고가 음수가 됨(216 + -4000)           -> 조치: 출처 체계에 증감량 재확인 요청
 E08 권한 없는 역할       권한 없는 역할 '병사'                    -> 조치: 지휘관 또는 군수참모 계정으로 접속
 E01 빈 이벤트          필수 항목 'id' 누락                    -> 조치: 이벤트에 'id' 값을 넣어 다시 전송
예외 10종 중 로그 기록 10건(기록률 100%), 크래시 0건
로그 예시: {"level": "ERROR", "code": "E01", "msg": "필수 항목 'delta' 누락", "guide": "이벤트에 'delta' 값을 넣어 다시 전송"}
목표(크래시 0건, 에러 로그 기록률 100%): 달성
```
</details>


---

## 🎯 미션 07. 체계 간 데이터 충돌 시 마스터 데이터 확정을 위한 데이터관리관 HITL 승인 설계
**소요시간**: 약 3.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

데이터 충돌 시 마스터 데이터 확정을 위한 데이터관리관 HITL 승인을 설계한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **체계 간 데이터 충돌 시 시스템은 후보만 제안하고 데이터관리관 승인으로만 마스터 데이터를 확정하는 사람 승인 체계를 설계한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **충돌 건마다 다수값 후보와 근거를 제안으로 등록하고, 데이터관리관이 근거를 적어 승인하거나 반려하게 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/workflow/dat09_approval_system.py` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 군 보안 규정 및 행정 절차 부합성

#### 미션 PASS 조건

- 정량 기준: 이력 기록 무결성 100%, 승인 권한 우회 시도 차단율 100%


- 진행 기준: 결재선 승인/반려 시뮬레이션 및 이력 조회 화면 구현


- 검증 방법: 권한별 접근 제어(RBAC) 테스트


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""체계 간 수량 충돌이 생기면 시스템이 마스터 값 후보와 근거를 제안하고 데이터관리관이 승인해야만 마스터를 확정하는 사람 승인 체계를 설계하며(고영향 건은 군수참모 추가 확인), 해시 체인 감사 이력의 무결성과 권한 우회 차단을 검증한다."""
import collections
import datetime
import hashlib
import json
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 이기종 군수정보체계 3종(A 보급, B 정비, C 탄약)의 재고 보고 생성기
# 실제 체계명·부대·품목·수량 없음. 같은 (품목, 부대) 재고를 세 체계가 서로 다른 코드·날짜 형식으로 보고한다.
# 기준 날짜: 2026-09-30(가상). 마스터(정답) 재고 TRUE 와 체계별 보고에는 의도적 불일치·결측이 섞여 있다.
# ---------------------------------------------------------------------------
N_PART, N_UNIT = 60, 12
CATS = ["급식", "피복", "유류", "정비소모", "수리부속", "탄약"]
PARTS = {p: {"name": f"가상품목{p:02d}", "cat": CATS[(p - 1) % 6]} for p in range(1, N_PART + 1)}
FMT = {"A": dict(part=lambda p: f"AP-{p:05d}", unit=lambda u: f"U{u:02d}", date=lambda d: d.strftime("%Y%m%d")),
       "B": dict(part=lambda p: f"{p:05d}-X", unit=lambda u: f"가상{u:02d}대대", date=lambda d: d.strftime("%Y-%m-%d")),
       "C": dict(part=lambda p: f"P{p}", unit=lambda u: str(u), date=lambda d: d.strftime("%d/%m/%Y"))}
BASE_DATE = datetime.date(2026, 9, 30)


def make_systems(seed=9, mismatch=0.10, missing=0.03):
    """TRUE(정답 재고)와 체계별 원본 보고 레코드(체계 고유 형식)를 만든다."""
    rng = np.random.default_rng(seed)
    true = {(p, u): int(rng.integers(20, 500)) for p in PARTS for u in range(1, N_UNIT + 1)}
    out = {s: [] for s in "ABC"}
    truth_diff = {}                                   # (체계, p, u) -> 불일치 원인(정답)
    for (p, u), q in true.items():
        for s in "ABC":
            qty, cause = q, None
            r = rng.random()
            if r < mismatch / 3:                      # 입력 오류(자릿수/오타)
                qty, cause = q + int(rng.choice([-1, 1]) * rng.integers(10, 60)), "입력 오류"
            elif r < 2 * mismatch / 3:                # 반영 지연(전일 값)
                qty, cause = q + int(rng.integers(5, 40)), "반영 지연"
            elif r < mismatch:                        # 단위 오류(10배)
                qty, cause = q * 10, "단위 오류"
            d = BASE_DATE - datetime.timedelta(days=int(cause == "반영 지연"))
            if rng.random() < missing:                # 결측
                qty, cause = None, "결측"
            if cause:
                truth_diff[(s, p, u)] = cause
            out[s].append({"part": FMT[s]["part"](p), "unit": FMT[s]["unit"](u), "date": FMT[s]["date"](d), "qty": qty})
    return true, out, truth_diff


def parse_part(code):
    # [TODO : 학생 실습 7] 체계 간 데이터 충돌 시 마스터 데이터 확정을 위한 데이터관리관 HITL 승인 설계 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def parse_unit(code):
    # [TODO : 학생 실습 7] 체계 간 데이터 충돌 시 마스터 데이터 확정을 위한 데이터관리관 HITL 승인 설계 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def parse_date(s):
    if "/" in s:
        d, m, y = s.split("/"); return datetime.date(int(y), int(m), int(d))
    if "-" in s:
        return datetime.date(*map(int, s.split("-")))
    return datetime.date(int(s[:4]), int(s[4:6]), int(s[6:]))

true, raw, tdiff = make_systems()
# [교육용가상자료] 가상 직책·인물(가명)
USERS = {"sys_bot": "시스템", "kim": "체계담당자", "lee": "체계담당자", "park": "데이터관리관", "choi": "군수참모", "han": "열람자"}
MASTER, PROPOSALS, AUDIT = {}, {}, []

def h(prev, rec):
    return hashlib.sha256((prev + json.dumps(rec, ensure_ascii=False, sort_keys=True, default=str)).encode()).hexdigest()[:16]

def log(key, user, action, result, note=""):
    rec = {"seq": len(AUDIT) + 1, "key": str(key), "user": user, "role": USERS.get(user, "미등록"), "action": action, "result": result, "note": note}
    rec["hash"] = h(AUDIT[-1]["hash"] if AUDIT else "GENESIS", {k: v for k, v in rec.items() if k != "hash"})
    AUDIT.append(rec)

def verify():
    prev, ok = "GENESIS", 0
    for r in AUDIT:
        ok += r["hash"] == h(prev, {k: v for k, v in r.items() if k != "hash"}); prev = r["hash"]
    return ok, len(AUDIT)

def block(key, user, action, why):
    log(key, user, action, "차단", why); return False

def propose(key, user, cand, evidence, old):
    if USERS.get(user) not in ("시스템", "체계담당자"):
        return block(key, user, "제안", "제안 권한 없음")
    high = abs(cand - old) >= 100 or abs(cand - old) / max(old, 1) >= 0.5       # 고영향 기준(가상)
    PROPOSALS[key] = {"cand": cand, "old": old, "by": user, "high": high, "staff_ok": False, "status": "대기"}
    log(key, user, "제안", "접수", f"후보 {cand}, 근거 {evidence}"); return True

def staff_confirm(key, user, note):
    p = PROPOSALS.get(key)
    if USERS.get(user) != "군수참모" or p is None or not p["high"]:
        return block(key, user, "참모확인", "고영향 건에 대한 군수참모만 가능")
    if len(note.strip()) < 5:
        return block(key, user, "참모확인", "확인 의견 5자 이상 필요")
    p["staff_ok"] = True; log(key, user, "참모확인", "완료", note); return True

def approve(key, user, note):
    p = PROPOSALS.get(key)
    if p is None:
        return block(key, user, "승인", "제안 없는 키")
    if p["status"] != "대기":
        return block(key, user, "승인", f"이미 처리됨({p['status']})")
    if USERS.get(user) != "데이터관리관":
        return block(key, user, "승인", f"권한 없음: 역할 '{USERS.get(user, '미등록')}'은(는) 마스터 확정 불가(자동 확정 금지)")
    if p["by"] == user:
        return block(key, user, "승인", "제안자 본인 승인 금지")
    if len(note.strip()) < 5:
        return block(key, user, "승인", "승인 근거 5자 이상 필요")
    if p["high"] and not p["staff_ok"]:
        return block(key, user, "승인", "고영향 건은 군수참모 확인 선행 필요")
    MASTER[key] = p["cand"]; p["status"] = "확정"; log(key, user, "승인", "확정", note); return True

def reject(key, user, note):
    p = PROPOSALS.get(key)
    if USERS.get(user) != "데이터관리관" or p is None or p["status"] != "대기" or len(note.strip()) < 5:
        return block(key, user, "반려", "반려 조건 불충족")
    p["status"] = "반려"; log(key, user, "반려", "반려", note); return True

# 1) 충돌 키 찾기: 세 체계 값이 서로 다르면 다수값을 후보로 제안
vals = collections.defaultdict(dict)
for s in "ABC":
    for r in raw[s]:
        vals[(parse_part(r["part"]), parse_unit(r["unit"]))][s] = r["qty"]
confl = []
for k, v in vals.items():
    have = [x for x in v.values() if x is not None]
    if len(set(have)) > 1:
        top, n = collections.Counter(have).most_common(1)[0]
        if n >= 2:
            confl.append((k, top, v["A"] if v["A"] is not None else top))
confl = sorted(confl)[:24]
print(f"[충돌 건] 다수값 후보가 있는 충돌 {len(confl)}건을 처리 대상으로 선정 [교육용가상자료]")

for i, (k, cand, old) in enumerate(confl):
    propose(k, "sys_bot" if i % 2 else "kim", cand, "다수 체계 일치", old)
    if PROPOSALS[k]["high"]:
        staff_confirm(k, "choi", "수량 차이가 커서 출처 체계 대조 후 확인")
    if i == 3:
        reject(k, "park", "후보가 오래된 보고와 같아 보류 요청"); continue
    approve(k, "park", "체계 간 다수값 및 원인 분류 검토 완료")
acc = np.mean([MASTER[k] == true[k] for k in MASTER])
print(f"확정 {len(MASTER)}건, 반려 {sum(p['status'] == '반려' for p in PROPOSALS.values())}건, 고영향(군수참모 확인 필요) {sum(p['high'] for p in PROPOSALS.values())}건, 확정 값이 가상 정답과 일치 {acc:.0%}")

# 2) 권한 우회 시도 10종
k0 = confl[0][0]
pend = confl[-1][0]
PROPOSALS[pend].update(status="대기"); MASTER.pop(pend, None)
hi = next(k for k, p in PROPOSALS.items() if p["high"]) if any(p["high"] for p in PROPOSALS.values()) else pend
PROPOSALS[hi].update(status="대기", staff_ok=False); MASTER.pop(hi, None)
total = blocked = 0
print("\n[권한 우회 시도 결과]")
def attempt(name, fn):
    global total, blocked
    n = len(AUDIT); fn(); r = AUDIT[-1]
    ok = len(AUDIT) == n + 1 and r["result"] == "차단"
    total += 1; blocked += ok
    print(f" - {name}: {r['result']} ({r['note']})")
attempt("시스템이 자동으로 마스터 확정", lambda: approve(pend, "sys_bot", "자동 처리 결과를 확정함"))
attempt("체계담당자가 직접 확정", lambda: approve(pend, "lee", "제가 직접 확정합니다 확인"))
attempt("열람자가 확정", lambda: approve(pend, "han", "열람자 권한으로 확정 시도"))
attempt("미등록 사용자 확정", lambda: approve(pend, "hacker", "외부 계정 확정 시도 확인"))
attempt("승인 근거 없이 확정", lambda: approve(pend, "park", ""))
attempt("제안 없는 키 확정", lambda: approve((99, 99), "park", "존재하지 않는 제안 승인 시도"))
attempt("고영향 건을 군수참모 확인 없이 확정", lambda: approve(hi, "park", "참모 확인 단계를 건너뛰고 승인"))
attempt("제안자 본인(관리관)이 승인", lambda: (PROPOSALS.update({pend: {**PROPOSALS[pend], "by": "park"}}), approve(pend, "park", "본인 제안을 본인이 승인"))[1])
attempt("체계담당자가 참모확인 대행", lambda: staff_confirm(hi, "kim", "참모 대신 확인했습니다 확인"))
PROPOSALS[pend].update(by="kim"); approve(pend, "park", "다수 체계 일치 확인 후 정상 승인")
attempt("이미 확정된 건 재승인", lambda: approve(pend, "park", "이미 확정된 건을 다시 승인"))
print(f"우회 시도 {total}건 중 차단 {blocked}건({blocked / total:.0%}), 자동 확정 0건")

# 3) 감사 이력 무결성 + 위조 시험
ok, tot = verify()
print(f"\n감사 이력 {tot}건 해시 체인 검증 {ok}/{tot}({ok / tot:.0%})")
j = next(i for i, r in enumerate(AUDIT) if r["result"] == "차단")
keep = AUDIT[j]["result"]; AUDIT[j]["result"] = "확정"
ok2, _ = verify(); AUDIT[j]["result"] = keep
print(f"위조 시험: {j + 1}번 기록 변조 -> 검증 {ok2}/{tot} (변조 탐지 {'성공' if ok2 < tot else '실패'}), 원복 후 {verify()[0]}/{tot}")
df = pd.DataFrame(AUDIT)
print("[행위별 결과 요약]")
print(df.groupby(["action", "result"]).size().to_string())
print(f"목표(이력 무결성 100%, 우회 차단율 100%): {'달성' if ok == tot and blocked == total and ok2 < tot else '미달'}")
print("마스터 확정은 데이터관리관 승인으로만 가능하다. 실제 보안·행정 절차는 부대 규정과 지휘계통에 따른다(학습용 모의).")


<details>
<summary><b>💡 [참고] 미션 07 예상 실행 결과 (Output)</b></summary>

```text
[충돌 건] 다수값 후보가 있는 충돌 24건을 처리 대상으로 선정 [교육용가상자료]
확정 23건, 반려 1건, 고영향(군수참모 확인 필요) 4건, 확정 값이 가상 정답과 일치 100%

[권한 우회 시도 결과]
 - 시스템이 자동으로 마스터 확정: 차단 (권한 없음: 역할 '시스템'은(는) 마스터 확정 불가(자동 확정 금지))
 - 체계담당자가 직접 확정: 차단 (권한 없음: 역할 '체계담당자'은(는) 마스터 확정 불가(자동 확정 금지))
 - 열람자가 확정: 차단 (권한 없음: 역할 '열람자'은(는) 마스터 확정 불가(자동 확정 금지))
 - 미등록 사용자 확정: 차단 (권한 없음: 역할 '미등록'은(는) 마스터 확정 불가(자동 확정 금지))
 - 승인 근거 없이 확정: 차단 (승인 근거 5자 이상 필요)
 - 제안 없는 키 확정: 차단 (제안 없는 키)
 - 고영향 건을 군수참모 확인 없이 확정: 차단 (고영향 건은 군수참모 확인 선행 필요)
 - 제안자 본인(관리관)이 승인: 차단 (제안자 본인 승인 금지)
 - 체계담당자가 참모확인 대행: 차단 (고영향 건에 대한 군수참모만 가능)
 - 이미 확정된 건 재승인: 차단 (이미 처리됨(확정))
우회 시도 10건 중 차단 10건(100%), 자동 확정 0건

감사 이력 63건 해시 체인 검증 63/63(100%)
위조 시험: 53번 기록 변조 -> 검증 62/63 (변조 탐지 성공), 원복 후 63/63
[행위별 결과 요약]
action  result
반려      반려         1
승인      차단         9
        확정        24
제안      접수        24
참모확인    완료         4
        차단         1
목표(이력 무결성 100%, 우회 차단율 100%): 달성
마스터 확정은 데이터관리관 승인으로만 가능하다. 실제 보안·행정 절차는 부대 규정과 지휘계통에 따른다(학습용 모의).
```
</details>


---

## 🎯 미션 08. 수만 건의 이기종 군수 트랜잭션 동시 유입 시뮬레이션 부하 테스트
**소요시간**: 약 3.5시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

수만 건의 이기종 군수 트랜잭션 동시 유입 부하 테스트를 수행한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **세 체계에서 수만 건의 트랜잭션이 뒤섞여 유입되는 상황을 모의해 부하 테스트를 수행한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **체계별 코드 형식이 다른 트랜잭션 5천~5만 건을 만들고 중복 1%, 형식 오류 0.4%, 일시 장애 주입하여 처리 지연 및 복원력을 시험하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/pipeline/dat09_batch_runner.py` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 수작업 대비 업무 소요 시간 80% 이상 절감 효과

#### 미션 PASS 조건

- 정량 기준: 전체 배치 파이프라인 실행 시간 5분 이내, 자동화 성공률 99% 이상


- 진행 기준: 스케줄러 연동 테스트 및 자동 실행 확인


- 검증 방법: 실제 월간 업무 데이터 1년치 일괄 배치 시뮬레이션


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""세 체계에서 수만 건의 재고 변동 트랜잭션이 뒤섞여 유입되는 상황을 모의해 표준화·중복 제거·오류 격리·마이크로 배치 반영을 수행하고, 규모별 처리 시간과 자동화 성공률, 수작업 대비 절감률을 측정하는 부하 테스트를 실행한다."""
import datetime
import time
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 이기종 군수정보체계 3종(A 보급, B 정비, C 탄약)의 재고 보고 생성기
# 실제 체계명·부대·품목·수량 없음. 같은 (품목, 부대) 재고를 세 체계가 서로 다른 코드·날짜 형식으로 보고한다.
# 기준 날짜: 2026-09-30(가상). 마스터(정답) 재고 TRUE 와 체계별 보고에는 의도적 불일치·결측이 섞여 있다.
# ---------------------------------------------------------------------------
N_PART, N_UNIT = 60, 12
CATS = ["급식", "피복", "유류", "정비소모", "수리부속", "탄약"]
PARTS = {p: {"name": f"가상품목{p:02d}", "cat": CATS[(p - 1) % 6]} for p in range(1, N_PART + 1)}
FMT = {"A": dict(part=lambda p: f"AP-{p:05d}", unit=lambda u: f"U{u:02d}", date=lambda d: d.strftime("%Y%m%d")),
       "B": dict(part=lambda p: f"{p:05d}-X", unit=lambda u: f"가상{u:02d}대대", date=lambda d: d.strftime("%Y-%m-%d")),
       "C": dict(part=lambda p: f"P{p}", unit=lambda u: str(u), date=lambda d: d.strftime("%d/%m/%Y"))}
BASE_DATE = datetime.date(2026, 9, 30)


def make_systems(seed=9, mismatch=0.10, missing=0.03):
    """TRUE(정답 재고)와 체계별 원본 보고 레코드(체계 고유 형식)를 만든다."""
    rng = np.random.default_rng(seed)
    true = {(p, u): int(rng.integers(20, 500)) for p in PARTS for u in range(1, N_UNIT + 1)}
    out = {s: [] for s in "ABC"}
    truth_diff = {}                                   # (체계, p, u) -> 불일치 원인(정답)
    for (p, u), q in true.items():
        for s in "ABC":
            qty, cause = q, None
            r = rng.random()
            if r < mismatch / 3:                      # 입력 오류(자릿수/오타)
                qty, cause = q + int(rng.choice([-1, 1]) * rng.integers(10, 60)), "입력 오류"
            elif r < 2 * mismatch / 3:                # 반영 지연(전일 값)
                qty, cause = q + int(rng.integers(5, 40)), "반영 지연"
            elif r < mismatch:                        # 단위 오류(10배)
                qty, cause = q * 10, "단위 오류"
            d = BASE_DATE - datetime.timedelta(days=int(cause == "반영 지연"))
            if rng.random() < missing:                # 결측
                qty, cause = None, "결측"
            if cause:
                truth_diff[(s, p, u)] = cause
            out[s].append({"part": FMT[s]["part"](p), "unit": FMT[s]["unit"](u), "date": FMT[s]["date"](d), "qty": qty})
    return true, out, truth_diff


def parse_part(code):
    # [TODO : 학생 실습 8] 수만 건의 이기종 군수 트랜잭션 동시 유입 시뮬레이션 부하 테스트 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def parse_unit(code):
    # [TODO : 학생 실습 8] 수만 건의 이기종 군수 트랜잭션 동시 유입 시뮬레이션 부하 테스트 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def parse_date(s):
    if "/" in s:
        d, m, y = s.split("/"); return datetime.date(int(y), int(m), int(d))
    if "-" in s:
        return datetime.date(*map(int, s.split("-")))
    return datetime.date(int(s[:4]), int(s[4:6]), int(s[6:]))

def make_tx(n, seed=8):
    """n건 트랜잭션: 체계별 고유 코드 형식, 도착 시각 뒤섞임, 중복 1%, 형식 오류 0.4%."""
    rng = np.random.default_rng(seed)
    sysn = rng.choice(list("ABC"), n, p=[0.45, 0.30, 0.25])
    p, u = rng.integers(1, N_PART + 1, n), rng.integers(1, N_UNIT + 1, n)
    delta = rng.integers(-20, 21, n)
    part = np.where(sysn == "A", [f"AP-{x:05d}" for x in p], np.where(sysn == "B", [f"{x:05d}-X" for x in p], [f"P{x}" for x in p]))
    unit = np.where(sysn == "A", [f"U{x:02d}" for x in u], np.where(sysn == "B", [f"가상{x:02d}대대" for x in u], [str(x) for x in u]))
    tx = pd.DataFrame({"id": [f"TX{i:06d}" for i in range(n)], "sys": sysn, "part": part, "unit": unit, "delta": delta,
                       "ts": np.sort(rng.integers(0, 3600 * 1000, n))})
    dup = tx.sample(int(n * 0.01), random_state=seed)
    tx = pd.concat([tx, dup]).sample(frac=1, random_state=seed).reset_index(drop=True)          # 중복 포함, 도착 순서 뒤섞기
    bad = rng.choice(len(tx), int(len(tx) * 0.004), replace=False)
    tx.loc[bad, "part"] = "??-" + tx.loc[bad, "part"]                                           # 형식 오류 주입
    return tx


def process(tx, batch=2000, flaky=()):
    """마이크로 배치 처리. flaky에 든 배치 번호는 첫 시도에서 일시 장애가 나며 재시도로 복구한다."""
    stock = {(p, u): 0 for p in PARTS for u in range(1, N_UNIT + 1)}
    seen, stat = set(), {"ok": 0, "dup": 0, "quarantine": 0, "retry": 0}
    for b, i in enumerate(range(0, len(tx), batch)):
        chunk = tx.iloc[i:i + batch]
        for attempt in (1, 2, 3):
            if b in flaky and attempt == 1:
                stat["retry"] += 1; continue                                   # 일시 장애: 같은 배치를 다시 시도
            e1 = chunk["part"].str.extract(r"^(?:AP-(\d+)|(\d+)-X|P(\d+))$")
            e2 = chunk["unit"].str.extract(r"^(?:U(\d+)|가상(\d+)대대|(\d+))$")
            p, u = e1[0].fillna(e1[1]).fillna(e1[2]), e2[0].fillna(e2[1]).fillna(e2[2])
            valid = p.notna() & u.notna()
            isdup = chunk["id"].isin(seen) | chunk["id"].duplicated()
            stat["quarantine"] += int((~valid).sum()); stat["dup"] += int((valid & isdup).sum())
            good = chunk[valid & ~isdup]
            seen.update(good["id"])
            pp, uu = p[good.index].astype(int), u[good.index].astype(int)
            for k, d in pd.DataFrame({"p": pp, "u": uu, "d": good["delta"]}).groupby(["p", "u"])["d"].sum().items():
                stock[k] += int(d)
            stat["ok"] += len(good)
            break
    return stock, stat


print("[규모별 부하 테스트] 체계 3종 트랜잭션 동시 유입 모의 [교육용가상자료]")
res = []
for n in (5000, 20000, 50000):
    tx = make_tx(n)
    t0 = time.perf_counter(); stock, st = process(tx, flaky=(1, 3)); dt = time.perf_counter() - t0
    res.append({"유입 건수": len(tx), "처리 시간(초)": round(dt, 2), "초당 처리": int(len(tx) / dt), **st})
    succ = (st["ok"] + st["dup"]) / len(tx)
    res[-1]["자동 처리율"] = f"{succ:.2%}"
print(pd.DataFrame(res).to_string(index=False))

# 정합성: 마지막 규모에서 일괄(벡터) 계산 결과와 마이크로 배치 결과 비교
tx2 = tx[~tx["part"].str.startswith("??")].drop_duplicates("id")
p = tx2["part"].str.extract(r"(\d+)")[0].astype(int)
u = tx2["unit"].str.extract(r"(\d+)")[0].astype(int)
ref = tx2.assign(p=p, u=u).groupby(["p", "u"])["delta"].sum()
stock, st = process(tx, flaky=(1, 3))
mism = sum(stock[(a, b)] != int(v) for (a, b), v in ref.items())
print(f"\n정합성: 일괄 계산과 배치 결과 비교 {len(ref)}개 키 중 불일치 {mism}건")
n = len(tx); success = (st["ok"] + st["dup"]) / n
print(f"전체 {n}건: 정상 반영 {st['ok']}, 중복 제거 {st['dup']}, 형식 오류 격리 {st['quarantine']}(담당자 확인), 일시 장애 재시도 {st['retry']}회")
total_s = res[-1]["처리 시간(초)"]
print(f"자동화 성공률 {success:.2%} (격리 건은 사람이 확인), 최대 규모 처리 시간 {total_s}초(목표 5분 이내: {'달성' if total_s < 300 else '미달'})")
manual = n * 0.4                                                  # 가정: 수작업 확인·입력 0.4분/건
auto = n * 0.01 + st["quarantine"] * 4 + total_s / 60             # 가정: 표본 검토 0.01분/건, 격리 건 수동 4분
print(f"업무 소요(가정값): 수작업 {manual:.0f}분 vs 자동화 {auto:.0f}분 -> 절감 {1 - auto / manual:.1%}")
print(f"\n목표(5분 이내, 자동화 성공률 99% 이상, 수작업 대비 80% 이상 절감): "
      f"{'달성' if total_s < 300 and success >= 0.99 and 1 - auto / manual >= 0.8 and mism == 0 else '미달'}")


<details>
<summary><b>💡 [참고] 미션 08 예상 실행 결과 (Output)</b></summary>

```text
[규모별 부하 테스트] 체계 3종 트랜잭션 동시 유입 모의 [교육용가상자료]
 유입 건수  처리 시간(초)  초당 처리    ok  dup  quarantine  retry 자동 처리율
  5050      0.04 114205  4982   48          20      1 99.60%
 20200      0.20 100425 19921  199          80      2 99.60%
 50500      0.69  73706 49801  497         202      2 99.60%

정합성: 일괄 계산과 배치 결과 비교 720개 키 중 불일치 0건
전체 50500건: 정상 반영 49801, 중복 제거 497, 형식 오류 격리 202(담당자 확인), 일시 장애 재시도 2회
자동화 성공률 99.60% (격리 건은 사람이 확인), 최대 규모 처리 시간 0.69초(목표 5분 이내: 달성)
업무 소요(가정값): 수작업 20200분 vs 자동화 1313분 -> 절감 93.5%

목표(5분 이내, 자동화 성공률 99% 이상, 수작업 대비 80% 이상 절감): 달성
```
</details>


---

## 🎯 미션 09. AAR 기반 데이터 연동 누락 분석 및 매핑 테이블 파라미터 재튜닝
**소요시간**: 약 3.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

AAR 기반 데이터 연동 누락을 분석하고 매핑 파라미터를 재튜닝한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **AAR에서 드러난 데이터 연동 누락 5유형을 재현·분석하고 매핑 규칙과 판정 파라미터를 재튜닝한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **코드 자릿수, 부대명, 날짜 형식, 결측 표기, 신규 품목의 5유형 변형을 가상 보고에 심어 매핑 v1의 누락을 재현 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `docs/dat09_uat_feedback_report.md 및 패치 내역` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 실무 업무 생산성 기여도 및 현장 적응성

#### 미션 PASS 조건

- 정량 기준: UAT 체크리스트 만족도 90점 이상, 개선 요구 5건 반영 완료


- 진행 기준: UAT 결과 보고서 작성 및 최종 버그 픽스 완료


- 검증 방법: 실제 업무 시나리오 기반 최종 검수 테스트


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""AAR에서 드러난 체계 연동 누락 5유형(코드 자릿수 변형, 부대명 변형, 날짜 형식 변형, 결측 표기 변형, 신규 품목)을 재현해 매핑 규칙 v1의 누락을 분석하고, 허용 오차·지연 일수 파라미터를 격자 탐색으로 재튜닝한 v2와 UAT 체크리스트 점수로 개선 효과를 검증한다."""
import datetime
import itertools
import re
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 이기종 군수정보체계 3종(A 보급, B 정비, C 탄약)의 재고 보고 생성기
# 실제 체계명·부대·품목·수량 없음. 같은 (품목, 부대) 재고를 세 체계가 서로 다른 코드·날짜 형식으로 보고한다.
# 기준 날짜: 2026-09-30(가상). 마스터(정답) 재고 TRUE 와 체계별 보고에는 의도적 불일치·결측이 섞여 있다.
# ---------------------------------------------------------------------------
N_PART, N_UNIT = 60, 12
CATS = ["급식", "피복", "유류", "정비소모", "수리부속", "탄약"]
PARTS = {p: {"name": f"가상품목{p:02d}", "cat": CATS[(p - 1) % 6]} for p in range(1, N_PART + 1)}
FMT = {"A": dict(part=lambda p: f"AP-{p:05d}", unit=lambda u: f"U{u:02d}", date=lambda d: d.strftime("%Y%m%d")),
       "B": dict(part=lambda p: f"{p:05d}-X", unit=lambda u: f"가상{u:02d}대대", date=lambda d: d.strftime("%Y-%m-%d")),
       "C": dict(part=lambda p: f"P{p}", unit=lambda u: str(u), date=lambda d: d.strftime("%d/%m/%Y"))}
BASE_DATE = datetime.date(2026, 9, 30)


def make_systems(seed=9, mismatch=0.10, missing=0.03):
    """TRUE(정답 재고)와 체계별 원본 보고 레코드(체계 고유 형식)를 만든다."""
    rng = np.random.default_rng(seed)
    true = {(p, u): int(rng.integers(20, 500)) for p in PARTS for u in range(1, N_UNIT + 1)}
    out = {s: [] for s in "ABC"}
    truth_diff = {}                                   # (체계, p, u) -> 불일치 원인(정답)
    for (p, u), q in true.items():
        for s in "ABC":
            qty, cause = q, None
            r = rng.random()
            if r < mismatch / 3:                      # 입력 오류(자릿수/오타)
                qty, cause = q + int(rng.choice([-1, 1]) * rng.integers(10, 60)), "입력 오류"
            elif r < 2 * mismatch / 3:                # 반영 지연(전일 값)
                qty, cause = q + int(rng.integers(5, 40)), "반영 지연"
            elif r < mismatch:                        # 단위 오류(10배)
                qty, cause = q * 10, "단위 오류"
            d = BASE_DATE - datetime.timedelta(days=int(cause == "반영 지연"))
            if rng.random() < missing:                # 결측
                qty, cause = None, "결측"
            if cause:
                truth_diff[(s, p, u)] = cause
            out[s].append({"part": FMT[s]["part"](p), "unit": FMT[s]["unit"](u), "date": FMT[s]["date"](d), "qty": qty})
    return true, out, truth_diff


def parse_part(code):
    # [TODO : 학생 실습 9] AAR 기반 데이터 연동 누락 분석 및 매핑 테이블 파라미터 재튜닝 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def parse_unit(code):
    # [TODO : 학생 실습 9] AAR 기반 데이터 연동 누락 분석 및 매핑 테이블 파라미터 재튜닝 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def parse_date(s):
    if "/" in s:
        d, m, y = s.split("/"); return datetime.date(int(y), int(m), int(d))
    if "-" in s:
        return datetime.date(*map(int, s.split("-")))
    return datetime.date(int(s[:4]), int(s[4:6]), int(s[6:]))

true, raw, tdiff = make_systems()
rng = np.random.default_rng(9)

# 1) AAR 사례 재현: 일부 보고를 체계 담당자가 다르게 쓴 형태로 바꾼다(변형 5유형)
VAR = {"코드 자릿수 변형": [], "부대명 변형": [], "날짜 형식 변형": [], "결측 표기 변형": [], "신규 품목": []}
def pick(s, n, f=lambda r: True):
    idx = [i for i, r in enumerate(raw[s]) if f(r) and not any(i == j for v in VAR.values() for (s2, j) in v if s2 == s)]
    return [(s, int(i)) for i in rng.choice(idx, n, replace=False)]
for s, i in pick("A", 25):
    raw[s][i]["part"] = "AP-" + raw[s][i]["part"][-4:]; VAR["코드 자릿수 변형"].append((s, i))
for s, i in pick("B", 30):
    raw[s][i]["unit"] = "가상" + str(int(raw[s][i]["unit"][2:4])) + "대대"; VAR["부대명 변형"].append((s, i))
for s, i in pick("C", 30):
    d, m, y = raw[s][i]["date"].split("/"); raw[s][i]["date"] = f"{d}-{m}-{y}"; VAR["날짜 형식 변형"].append((s, i))
for s, i in pick("A", 20, lambda r: r["qty"] is not None):
    raw[s][i]["qty"] = "N/A"; VAR["결측 표기 변형"].append((s, i))
for s, i in pick("B", 15):
    raw[s][i]["part"] = "00061-X"; VAR["신규 품목"].append((s, i))
for s in "ABC":                                                      # 반올림 차이 등 정상 변동(+-1)
    for i in rng.choice(len(raw[s]), 60, replace=False):
        if isinstance(raw[s][i]["qty"], int):
            raw[s][i]["qty"] += int(rng.choice([-1, 1]))

# 2) 매핑 규칙 v1(엄격) / v2(보정)
PAT1 = {"part": r"^(?:AP-(\d{5})|(\d{5})-X|P(\d+))$", "unit": r"^(?:U(\d{2})|가상(\d{2})대대|(\d+))$"}
PAT2 = {"part": r"^(?:AP-(\d{4,5})|(\d{5})-X|P(\d+))$", "unit": r"^(?:U(\d{2})|가상(\d{1,2})대대|(\d+))$"}
def g(pat, s):
    m = re.match(pat, s)
    return int(next(x for x in m.groups() if x)) if m else None
def pdate(s, v2):
    for f in ("%Y%m%d", "%Y-%m-%d", "%d/%m/%Y") + (("%d-%m-%Y",) if v2 else ()):
        try:
            return datetime.datetime.strptime(s, f).date()
        except ValueError:
            pass
    return None
def link(v2):
    pat, out, unlinked, held = (PAT2 if v2 else PAT1), [], 0, 0
    for s in "ABC":
        for i, r in enumerate(raw[s]):
            p, u, d = g(pat["part"], r["part"]), g(pat["unit"], r["unit"]), pdate(r["date"], v2)
            q = r["qty"]
            if q == "N/A":
                q = None if v2 else "ERR"
            if p is None or u is None or d is None or q == "ERR":
                unlinked += 1; continue
            if p > N_PART and v2:
                held += 1; continue                                  # v2: 신규 품목은 조용히 버리지 않고 '신규 품목 신청' 대상으로 보류
            if p > N_PART:
                unlinked += 1; continue
            out.append({"sys": s, "p": p, "u": u, "date": d, "qty": q})
    return pd.DataFrame(out), unlinked, held
d1, un1, h1 = link(False); d2, un2, h2 = link(True)
total = sum(len(v) for v in raw.values())
print(f"[AAR 연동 누락 분석] 전체 보고 {total}건 [교육용가상자료]")
print(f"매핑 v1: 연동 {len(d1)}건, 누락 {un1}건({un1 / total:.1%}) | 매핑 v2: 연동 {len(d2)}건, 누락 {un2}건, 신규 품목 보류 {h2}건")
# 사례별 v1 누락 건수는 직접 재판정
def lost(s, i, v2):
    r = raw[s][i]; pat = PAT2 if v2 else PAT1
    p, u, d = g(pat["part"], r["part"]), g(pat["unit"], r["unit"]), pdate(r["date"], v2)
    return p is None or u is None or d is None or (not v2 and (r["qty"] == "N/A" or p > N_PART))
tab = pd.DataFrame([{"사례": k, "발생": len(v), "v1 누락": sum(lost(s, i, False) for s, i in v), "v2 누락": sum(lost(s, i, True) for s, i in v)} for k, v in VAR.items()])
print(tab.to_string(index=False))

# 3) 불일치 판정 파라미터 재튜닝: 수량 허용 오차(%)와 허용 지연 일수의 격자 탐색(가상 정답 대비 F1)
gt = {(p, u) for (_, p, u), c in tdiff.items() if c != "결측"}
def detect(df, tol, lag):
    df = df.copy(); df["q"] = pd.to_numeric(df["qty"], errors="coerce")
    w = df.pivot_table(index=["p", "u"], columns="sys", values="q", aggfunc="first")
    med = w.median(axis=1)
    dev = w.sub(med, axis=0).abs().div(med.clip(lower=1), axis=0)
    late = df.assign(l=(BASE_DATE - df["date"]).map(lambda x: x.days)).pivot_table(index=["p", "u"], columns="sys", values="l", aggfunc="first")
    flag = ((dev > tol) | (late.reindex(w.index) > lag).fillna(False) & (dev > 0)).any(axis=1)
    return set(flag[flag].index)
def f1(found):
    tp = len(found & gt); pr = tp / max(len(found), 1); rc = tp / len(gt)
    return 2 * pr * rc / max(pr + rc, 1e-9), pr, rc
grid = []
for tol, lag in itertools.product((0.0, 0.005, 0.02, 0.05), (0, 1, 2)):
    grid.append({"허용오차": tol, "허용지연(일)": lag, **dict(zip(("F1", "정밀도", "재현율"), f1(detect(d2, tol, lag))))})
G = pd.DataFrame(grid).sort_values("F1", ascending=False)
best = G.iloc[0]
v1p = f1(detect(d2, 0.0, 0))
print("\n[파라미터 재튜닝 - 상위 3개 조합]")
print(G.head(3).round(3).to_string(index=False))
print(f"튜닝 전(오차 0, 지연 0) F1 {v1p[0]:.3f} -> 튜닝 후 F1 {best['F1']:.3f} (허용오차 {best['허용오차']}, 허용지연 {int(best['허용지연(일)'])}일)")

# 4) UAT 체크리스트(자동 점검 항목) + 개선 요구 5건
tests = [(f"{k} 사례 v2에서 해소", int(r["v2 누락"]) == 0) for k, r in tab.set_index("사례").iterrows()]
tests += [("v2 연동률이 v1보다 높음", len(d2) > len(d1)), ("신규 품목은 누락이 아니라 보류로 분류", h2 == 15), ("v1 연동 키가 v2에 모두 포함", set(zip(d1["sys"], d1["p"], d1["u"])) <= set(zip(d2["sys"], d2["p"], d2["u"]))),
          ("튜닝 후 F1이 튜닝 전보다 높음", best["F1"] > v1p[0]), ("재현율 95% 이상", best["재현율"] >= 0.95), ("정밀도 90% 이상", best["정밀도"] >= 0.9),
          ("연동 키가 (체계, 품목, 부대)로 유일", not d2.duplicated(["sys", "p", "u"]).any()), ("날짜 파싱 100%", d2["date"].notna().all()),
          ("v2 코드 재실행 결과 동일", len(link(True)[0]) == len(d2)), ("튜닝 격자 12조합 평가", len(G) == 12), ("v1 누락 건수가 사례 합계와 일치", int(tab["v1 누락"].sum()) == un1),
          ("원인-패치 대응표 작성", len(VAR) == 5), ("재튜닝 파라미터 기록", best["허용오차"] >= 0), ("개선 요구 5건 반영", all(tab["v2 누락"] == 0))]
passed = sum(ok for _, ok in tests)
score = 100 * passed / len(tests)
print(f"\n[UAT 체크리스트] {passed}/{len(tests)}항목 통과, 점수 {score:.0f}점")
for n, ok in tests:
    if not ok:
        print("  미통과:", n)
print("[개선 요구 반영 현황]")
PATCH = ["품목코드 4~5자리 허용", "부대명 1~2자리 허용", "날짜 형식 DD-MM-YYYY 추가", "결측 표기 N/A를 결측으로 처리", "신규 품목은 보류 후 신청 대상으로 표시"]
for i, ((k, r), pt) in enumerate(zip(tab.set_index("사례").iterrows(), PATCH), 1):
    print(f" 요구 {i}. {k}: {pt} -> {'반영 완료' if r['v2 누락'] == 0 else '미반영'}")
print(f"\n목표(UAT 90점 이상, 개선 요구 5건 반영): {'달성' if score >= 90 and all(tab['v2 누락'] == 0) else '미달'}")


<details>
<summary><b>💡 [참고] 미션 09 예상 실행 결과 (Output)</b></summary>

```text
[AAR 연동 누락 분석] 전체 보고 2160건 [교육용가상자료]
매핑 v1: 연동 2045건, 누락 115건(5.3%) | 매핑 v2: 연동 2145건, 누락 0건, 신규 품목 보류 15건
       사례  발생  v1 누락  v2 누락
코드 자릿수 변형  25     25      0
   부대명 변형  30     25      0
 날짜 형식 변형  30     30      0
 결측 표기 변형  20     20      0
    신규 품목  15     15      0

[파라미터 재튜닝 - 상위 3개 조합]
 허용오차  허용지연(일)    F1   정밀도   재현율
 0.02        0 0.972 0.972 0.972
 0.02        1 0.963 0.972 0.955
 0.02        2 0.963 0.972 0.955
튜닝 전(오차 0, 지연 0) F1 0.738 -> 튜닝 후 F1 0.972 (허용오차 0.02, 허용지연 0일)

[UAT 체크리스트] 19/19항목 통과, 점수 100점
[개선 요구 반영 현황]
 요구 1. 코드 자릿수 변형: 품목코드 4~5자리 허용 -> 반영 완료
 요구 2. 부대명 변형: 부대명 1~2자리 허용 -> 반영 완료
 요구 3. 날짜 형식 변형: 날짜 형식 DD-MM-YYYY 추가 -> 반영 완료
 요구 4. 결측 표기 변형: 결측 표기 N/A를 결측으로 처리 -> 반영 완료
 요구 5. 신규 품목: 신규 품목은 보류 후 신청 대상으로 표시 -> 반영 완료

목표(UAT 90점 이상, 개선 요구 5건 반영): 달성
```
</details>


---

## 🎯 미션 10. 이기종 군수 정보체계 데이터 통합 및 실시간 가시화 대시보드 최종 완성
**소요시간**: 약 3.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

이기종 군수 정보체계 데이터 통합 및 실시간 가시화 대시보드를 완성한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **수집부터 불일치 식별, 품질 지수, 마스터 승인 대기 분리, 상황판 가시화까지 한 번에 실행되는 통합 군수 상황판 도구를 완성한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **환경 점검 후 세 체계 자료를 수집·표준화하고 불일치를 식별해 품질 지수 산출하고 모델에 적용하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `dist/dat09_package/, README.md, docs/dat09_final_manual.md` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 상용 소프트웨어 수준의 완성도 및 조직 확산 가치

#### 미션 PASS 조건

- 정량 기준: 설치부터 실행까지 원클릭 설치 스크립트 완비, 매뉴얼 작성률 100%


- 진행 기준: 최종 배포본 생성 및 검수 완료


- 검증 방법: Clean 환경에서의 설치 및 초기화 테스트


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""환경 점검부터 3개 체계 자료 수집·표준화, 체계 간 불일치 식별, 품질 지수 산출, 마스터 후보의 데이터관리관 승인 대기 분리, 통합 군수 상황판 가시화까지 한 번에 실행하는 도구를 완성하고 설치 개요·매뉴얼 목차·자체 점검·브리핑 요약을 출력한다."""
import collections
import csv
import datetime
import importlib.util
import io
import json
import sys
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 이기종 군수정보체계 3종(A 보급, B 정비, C 탄약)의 재고 보고 생성기
# 실제 체계명·부대·품목·수량 없음. 같은 (품목, 부대) 재고를 세 체계가 서로 다른 코드·날짜 형식으로 보고한다.
# 기준 날짜: 2026-09-30(가상). 마스터(정답) 재고 TRUE 와 체계별 보고에는 의도적 불일치·결측이 섞여 있다.
# ---------------------------------------------------------------------------
N_PART, N_UNIT = 60, 12
CATS = ["급식", "피복", "유류", "정비소모", "수리부속", "탄약"]
PARTS = {p: {"name": f"가상품목{p:02d}", "cat": CATS[(p - 1) % 6]} for p in range(1, N_PART + 1)}
FMT = {"A": dict(part=lambda p: f"AP-{p:05d}", unit=lambda u: f"U{u:02d}", date=lambda d: d.strftime("%Y%m%d")),
       "B": dict(part=lambda p: f"{p:05d}-X", unit=lambda u: f"가상{u:02d}대대", date=lambda d: d.strftime("%Y-%m-%d")),
       "C": dict(part=lambda p: f"P{p}", unit=lambda u: str(u), date=lambda d: d.strftime("%d/%m/%Y"))}
BASE_DATE = datetime.date(2026, 9, 30)


def make_systems(seed=9, mismatch=0.10, missing=0.03):
    """TRUE(정답 재고)와 체계별 원본 보고 레코드(체계 고유 형식)를 만든다."""
    rng = np.random.default_rng(seed)
    true = {(p, u): int(rng.integers(20, 500)) for p in PARTS for u in range(1, N_UNIT + 1)}
    out = {s: [] for s in "ABC"}
    truth_diff = {}                                   # (체계, p, u) -> 불일치 원인(정답)
    for (p, u), q in true.items():
        for s in "ABC":
            qty, cause = q, None
            r = rng.random()
            if r < mismatch / 3:                      # 입력 오류(자릿수/오타)
                qty, cause = q + int(rng.choice([-1, 1]) * rng.integers(10, 60)), "입력 오류"
            elif r < 2 * mismatch / 3:                # 반영 지연(전일 값)
                qty, cause = q + int(rng.integers(5, 40)), "반영 지연"
            elif r < mismatch:                        # 단위 오류(10배)
                qty, cause = q * 10, "단위 오류"
            d = BASE_DATE - datetime.timedelta(days=int(cause == "반영 지연"))
            if rng.random() < missing:                # 결측
                qty, cause = None, "결측"
            if cause:
                truth_diff[(s, p, u)] = cause
            out[s].append({"part": FMT[s]["part"](p), "unit": FMT[s]["unit"](u), "date": FMT[s]["date"](d), "qty": qty})
    return true, out, truth_diff


def parse_part(code):
    # [TODO : 학생 실습 10] 이기종 군수 정보체계 데이터 통합 및 실시간 가시화 대시보드 최종 완성 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def parse_unit(code):
    # [TODO : 학생 실습 10] 이기종 군수 정보체계 데이터 통합 및 실시간 가시화 대시보드 최종 완성 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def parse_date(s):
    if "/" in s:
        d, m, y = s.split("/"); return datetime.date(int(y), int(m), int(d))
    if "-" in s:
        return datetime.date(*map(int, s.split("-")))
    return datetime.date(int(s[:4]), int(s[4:6]), int(s[6:]))

LOG = []
def stage(name):
    def deco(fn):
        def run(*a):
            t0 = time.perf_counter()
            try:
                r = fn(*a); LOG.append((name, "OK", time.perf_counter() - t0)); return r
            except Exception as ex:                       # 한 단계가 실패해도 도구는 멈추지 않고 로그를 남긴다
                LOG.append((name, f"FAIL {repr(ex)[:30]}", time.perf_counter() - t0)); return None
        return run
    return deco

INSTALL = ["1. 가상 환경 생성, numpy·pandas·matplotlib 설치", "2. 환경 점검(모듈 확인)", "3. 샘플 3개 체계 자료로 시험 실행", "4. run_cop() 실행", "5. 상황판·승인 대기 목록 확인"]
mods = ["numpy", "pandas", "matplotlib", "csv", "json"]
env_ok = sum(importlib.util.find_spec(m) is not None for m in mods)
print(f"[0] 환경 점검: Python {sys.version_info.major}.{sys.version_info.minor}, 필수 모듈 {env_ok}/{len(mods)} 확인 [교육용가상자료]")

true, raw, tdiff = make_systems()

@stage("수집")
def collect():
    a = "\n".join(["part,unit,date,qty"] + [f"{r['part']},{r['unit']},{r['date']},{'' if r['qty'] is None else r['qty']}" for r in raw["A"]])
    b = json.dumps([{"part": r["part"], "unit": r["unit"], "date": r["date"], "qty": r["qty"]} for r in raw["B"]], ensure_ascii=False)
    c = "\n".join(["part\tunit\tdate\tqty"] + [f"{r['part']}\t{r['unit']}\t{r['date']}\t{-1 if r['qty'] is None else r['qty']}" for r in raw["C"]])
    out = []
    for s, rows in (("A", list(csv.DictReader(io.StringIO(a)))), ("B", json.loads(b)), ("C", list(csv.DictReader(io.StringIO(c), delimiter="\t")))):
        for r in rows:
            q = r["qty"]
            q = None if q in ("", None, "-1", -1) else int(q)
            out.append({"sys": s, "p": parse_part(r["part"]), "u": parse_unit(r["unit"]), "date": parse_date(r["date"]), "qty": q})
    return pd.DataFrame(out)

@stage("불일치 식별")
def reconcile(df):
    w = df.pivot_table(index=["p", "u"], columns="sys", values="qty", aggfunc="first")
    rows = []
    for k, r in w.iterrows():
        have = r.dropna()
        top, n = collections.Counter(have.values).most_common(1)[0] if len(have) else (None, 0)
        diff = (have != top).any() or len(have) < 3
        rows.append({"p": k[0], "u": k[1], "mismatch": bool(diff), "cand": top if n >= 2 else None, "n_missing": 3 - len(have)})
    return pd.DataFrame(rows)

@stage("품질 지수")
def quality(df, rec):
    comp = df["qty"].notna().mean(); cons = 1 - rec["mismatch"].mean()
    tim = ((BASE_DATE - df["date"]).map(lambda d: d.days) <= 0).mean()
    return 100 * (0.3 * comp + 0.5 * cons + 0.2 * tim)

@stage("승인 대기 분리")
def queue(rec):
    q = rec[rec["mismatch"]].copy()
    q["status"] = np.where(q["cand"].notna(), "데이터관리관 승인 대기", "사람 판단 필요(후보 없음)")
    return q

@stage("상황판")
def cop(df):
    base = df[df["sys"] == "A"].dropna(subset=["qty"]).copy()
    base["cat"] = base["p"].map(lambda p: PARTS[p]["cat"])
    return base.groupby("cat")["qty"].sum(), base.groupby("u")["qty"].sum()

df = collect()
rec = reconcile(df)
qi = quality(df, rec)
q = queue(rec)
cat_tot, unit_tot = cop(df)

print(f"[1] 수집·표준화 {len(df)}건(3개 체계), 통합 키 {len(rec)}개")
print(f"[2] 체계 간 불일치 {int(rec['mismatch'].sum())}개({rec['mismatch'].mean():.1%}), 통합 데이터 품질 지수 {qi:.1f}점")
print("[3] 마스터 확정 대기 현황(자동 확정 없음):", q["status"].value_counts().to_dict())
print("[4] 통합 군수 상황판 - 분류별 재고(체계 A 기준):", cat_tot.astype(int).to_dict())
print("    재고 적은 부대 3곳:", unit_tot.nsmallest(3).astype(int).to_dict())
print("[5] 단계별 실행 로그:")
for n, st, t in LOG:
    print(f"   - {n:<8} {st:<5} {t * 1000:.1f}ms")
run_ms = sum(t for _, _, t in LOG) * 1000

fig, ax = plt.subplots(1, 3, figsize=(13, 3.8))
ax[0].bar([f"cat{i + 1}" for i in range(len(cat_tot))], cat_tot.values, color="tab:blue"); ax[0].set_title("Stock by category")
ax[1].bar(unit_tot.index, unit_tot.values, color="tab:gray"); ax[1].set_title("Stock by unit")
ax[2].bar(["pending", "no candidate"], [int((q["status"].str.startswith("데이터")).sum()), int((~q["status"].str.startswith("데이터")).sum())], color=["tab:orange", "tab:red"])
ax[2].set_title("Master approval queue")
plt.suptitle("DAT09 integrated logistics board (synthetic)"); plt.tight_layout(); plt.show()

MANUAL = ["1. 개요·설치", "2. 체계별 입력 규격", "3. 코드 변환표·매핑 규칙", "4. 불일치 처리 절차", "5. 마스터 승인(데이터관리관)", "6. 상황판 보는 법", "7. 오류 메시지 조치", "8. 부하·배치 운영", "9. 개선 이력·문의"]
checks = [("환경 점검 5/5", env_ok == len(mods)), ("모든 단계 OK", all(s == "OK" for _, s, _ in LOG)), ("통합 키 720개 생성", len(rec) == 720),
          ("불일치 식별 재현율 95% 이상", len({(r.p, r.u) for r in rec.itertuples() if r.mismatch} & {(p, u) for (_, p, u) in tdiff}) / len({(p, u) for (_, p, u) in tdiff}) >= 0.95),
          ("마스터 자동 확정 0건", not q["status"].eq("확정").any()), ("실행 시간 1분 이내", run_ms < 60000),
          ("매뉴얼 9장 작성(작성률 100%)", len(MANUAL) == 9), ("설치 절차 5단계 완비", len(INSTALL) == 5)]
print("\n[6] 자체 점검")
for n, ok in checks:
    print(f"   {'PASS' if ok else 'FAIL'}  {n}")
npass = sum(ok for _, ok in checks)
print("설치 스크립트 개요:", " / ".join(INSTALL))
print("매뉴얼 목차:", " | ".join(MANUAL))
asis, tobe = 550, 50
print(f"\n[브리핑 요약] 3개 체계 {len(df)}건을 {run_ms:.0f}ms에 통합. 불일치 {int(rec['mismatch'].sum())}개는 후보와 함께 데이터관리관 승인 대기로 분리(자동 확정 없음).")
print(f"품질 지수 {qi:.1f}점, 상황 파악 시간 {asis}분 -> {tobe}분({1 - tobe / asis:.0%} 절감 가정값).")
print(f"자체 점검 {npass}/{len(checks)} PASS. 목표(원클릭 설치·실행 완비, 매뉴얼 100%): {'달성' if npass == len(checks) else '미달'}")


<details>
<summary><b>💡 [참고] 미션 10 예상 실행 결과 (Output)</b></summary>

```text
[0] 환경 점검: Python 3.11, 필수 모듈 5/5 확인 [교육용가상자료]
[1] 수집·표준화 2160건(3개 체계), 통합 키 720개
[2] 체계 간 불일치 219개(30.4%), 통합 데이터 품질 지수 83.5점
[3] 마스터 확정 대기 현황(자동 확정 없음): {'데이터관리관 승인 대기': 198, '사람 판단 필요(후보 없음)': 21}
[4] 통합 군수 상황판 - 분류별 재고(체계 A 기준): {'급식': 33087, '수리부속': 42203, '유류': 40597, '정비소모': 38378, '탄약': 39855, '피복': 39222}
    재고 적은 부대 3곳: {9: 15552, 6: 15588, 3: 15797}
[5] 단계별 실행 로그:
   - 수집       OK    14.4ms
   - 불일치 식별   OK    147.9ms
   - 품질 지수    OK    1.7ms
   - 승인 대기 분리 OK    1.1ms
   - 상황판      OK    4.5ms

[6] 자체 점검
   PASS  환경 점검 5/5
   PASS  모든 단계 OK
   PASS  통합 키 720개 생성
   PASS  불일치 식별 재현율 95% 이상
   PASS  마스터 자동 확정 0건
   PASS  실행 시간 1분 이내
   PASS  매뉴얼 9장 작성(작성률 100%)
   PASS  설치 절차 5단계 완비
설치 스크립트 개요: 1. 가상 환경 생성, numpy·pandas·matplotlib 설치 / 2. 환경 점검(모듈 확인) / 3. 샘플 3개 체계 자료로 시험 실행 / 4. run_cop() 실행 / 5. 상황판·승인 대기 목록 확인
매뉴얼 목차: 1. 개요·설치 | 2. 체계별 입력 규격 | 3. 코드 변환표·매핑 규칙 | 4. 불일치 처리 절차 | 5. 마스터 승인(데이터관리관) | 6. 상황판 보는 법 | 7. 오류 메시지 조치 | 8. 부하·배치 운영 | 9. 개선 이력·문의

[브리핑 요약] 3개 체계 2160건을 170ms에 통합. 불일치 219개는 후보와 함께 데이터관리관 승인 대기로 분리(자동 확정 없음).
품질 지수 83.5점, 상황 파악 시간 550분 -> 50분(91% 절감 가정값).
자체 점검 8/8 PASS. 목표(원클릭 설치·실행 완비, 매뉴얼 100%): 달성
```
</details>
